# Hidden Payload Recovery from ode.pt
This notebook continues the payload recovery effort by extracting the adjacent transformer weight tensors, reconstructing hidden data with deinterleaving, and analyzing the extracted bitplane payloads.

## 1. Import Required Libraries
Load Python libraries for tensor handling, binary analysis, and file I/O.

In [12]:
import os
import struct
import binascii
import torch
import numpy as np

print('torch', torch.__version__)

torch 2.12.0+cu130


## 2. Load tensors and extract raw weight bytes
Open the checkpoint, extract the two suspicious weight tensors, and convert them to raw byte strings.

In [13]:
checkpoint_path = 'ode.pt'
assert os.path.exists(checkpoint_path), f'Missing checkpoint: {checkpoint_path}'
obj = torch.load(checkpoint_path, map_location='cpu', weights_only=False)
layer6 = 'transformer.h.6.mlp.c_proj.weight'
layer7 = 'transformer.h.7.mlp.c_proj.weight'
raw6 = obj['model'][layer6].numpy().tobytes()
raw7 = obj['model'][layer7].numpy().tobytes()
print(layer6, len(raw6))
print(layer7, len(raw7))
for name, raw in [(layer6, raw6), (layer7, raw7)]:
    offset = raw.find(b'ctf=')
    print(f'{name} ctf= offset: {offset}')

transformer.h.6.mlp.c_proj.weight 6553600
transformer.h.7.mlp.c_proj.weight 6553600
transformer.h.6.mlp.c_proj.weight ctf= offset: -1
transformer.h.7.mlp.c_proj.weight ctf= offset: 6507128


## 3. Reconstruct hidden payload by deinterleaving
Implement `deinterleave(raw6, raw7, block_size)` and reconstruct combined streams for block sizes 4 and 8. Search for payload markers in reconstructed blobs.

In [14]:
def deinterleave(a: bytes, b: bytes, block_size: int) -> bytes:
    out = bytearray()
    i = 0
    while i < len(a) and i < len(b):
        out.extend(a[i:i+block_size])
        out.extend(b[i:i+block_size])
        i += block_size
    return bytes(out)

def scan_blob(blob: bytes, label: str):
    markers = [b'ctf=', b'ctf{', b'flag{', b'PK', b'\x1f\x8b', b'-----BEGIN']
    print(f'--- scan {label} ---')
    for marker in markers:
        idx = blob.find(marker)
        if idx != -1:
            print(f'{marker!r} at {idx}')

for block in [4, 8]:
    blob = deinterleave(raw6, raw7, block)
    print(f'block={block} length={len(blob)}')
    scan_blob(blob, f'deinterleave_{block}')
    idx = blob.find(b'ctf=')
    if idx != -1:
        print('ctf= window:', blob[idx:idx+64])

block=4 length=13107200
--- scan deinterleave_4 ---
b'ctf=' at 13014260
b'PK' at 128085
b'\x1f\x8b' at 51960
ctf= window: b'ctf=\xcf\xe4\x15\xbe\xc5\xca\x14\xbe\xe3\x9d\xda\xbco\xcc\x85<\x98b\x85<\xaf?\xbe\xbd\xd4\xda\xab\xbdl\xec\t>\xc3\xacl:\x8e\xcc\x01\xbbkI\xae<u"\xa9\xbd\xd1Du\xbci\x94?>Dt\x14<'
block=8 length=13107200
--- scan deinterleave_8 ---
b'ctf=' at 13014264
b'PK' at 128089
b'\x1f\x8b' at 51956
ctf= window: b'ctf=\xc5\xca\x14\xbe\xe3\x9d\xda\xbc\x98b\x85<o\xcc\x85<\xaf?\xbe\xbd\xd4\xda\xab\xbd\xc3\xacl:l\xec\t>\x8e\xcc\x01\xbbkI\xae<\xd1Du\xbcu"\xa9\xbdi\x94?>Dt\x14<o\xd6\xaf<'


## 4. Extract candidate bitplane payloads
Extract bit planes from the deinterleaved payload and write candidate files for further analysis.

In [15]:
def bitplane_bytes(data: bytes, bit: int) -> bytes:
    bits = [(c >> bit) & 1 for c in data]
    out = bytearray()
    for i in range(0, len(bits) - (len(bits) % 8), 8):
        byte = 0
        for j in range(8):
            byte |= bits[i + j] << j
        out.append(byte)
    return bytes(out)

def printable_ratio(data: bytes) -> float:
    return sum(32 <= c < 127 for c in data) / len(data) if data else 0.0

for block in [4, 8]:
    blob = deinterleave(raw6, raw7, block)
    idx = blob.find(b'ctf=')
    if idx == -1:
        continue
    data = blob[idx:idx+65536]
    for bit in [1, 6]:
        out = bitplane_bytes(data, bit)
        filename = f'bitplane_{block}_{bit}.bin'
        with open(filename, 'wb') as f:
            f.write(out)
        print(f'wrote {filename} length={len(out)} printable={printable_ratio(out):.4f}')
        first_ascii = None
        current = bytearray()
        for c in out:
            if 32 <= c < 127:
                current.append(c)
            else:
                if len(current) >= 8:
                    first_ascii = bytes(current).decode('ascii', errors='ignore')
                    break
                current = bytearray()
        if first_ascii:
            print('first ascii run:', first_ascii)

wrote bitplane_4_1.bin length=8192 printable=0.6306
first ascii run: =3v:DtD/a6C&
wrote bitplane_4_6.bin length=8192 printable=0.7079
first ascii run: 7S#0S2q13R6vQ6sBd
wrote bitplane_8_1.bin length=8192 printable=0.6339
first ascii run: hEFQFSv#
wrote bitplane_8_6.bin length=8192 printable=0.7069
first ascii run: 3#5fWas#dt`


## 5. Analyze `bitplane_4_1.bin` as OpenPGP-like data
Inspect the extracted blob header, parse packet version and length fields, and verify the OpenPGP-like packet structure.

In [16]:
filename = 'bitplane_4_1.bin'
if os.path.exists(filename):
    data = open(filename, 'rb').read()
    print(filename, 'len=', len(data))
    print('first 32 bytes:', data[:32])
    if len(data) >= 3:
        tag = data[0]>>2
        new_format = bool(data[0] & 0x40)
        print('packet tag', tag, 'new_format', new_format)
    if len(data) >= 6:
        print('next bytes:', data[1:6])
    if len(data) >= 10:
        length_byte = data[1]
        print('length byte', length_byte)
        if length_byte < 192:
            print('declared length', length_byte)
        elif 192 <= length_byte <= 223:
            next_len = ((length_byte - 192) << 8) + data[2] + 192
            print('declared length', next_len)
        elif length_byte == 255:
            print('declared length', struct.unpack('>I', data[2:6])[0])
        else:
            print('partial/truncated length marker', length_byte)
else:
    print('missing', filename)

bitplane_4_1.bin len= 8192
first 32 bytes: b'\x95Z!g\x98Y\x02\x0ct\xe4\x05=3v:DtD/a6C&\x00\x05\x10s\x0673A\xa8'
packet tag 37 new_format False
next bytes: b'Z!g\x98Y'
length byte 90
declared length 90


## 6. Locate missing payload bytes and extend search window
Search for additional payload bytes beyond the current raw6/raw7 window, and test alternate alignments against the deinterleaved payload.

In [17]:
for block in [4, 8]:
    blob = deinterleave(raw6, raw7, block)
    idx = blob.find(b'ctf=')
    if idx == -1:
        continue
    window_start = max(0, idx - 4096)
    window_end = min(len(blob), idx + 65536)
    window = blob[window_start:window_end]
    print(f'block={block} window bytes {window_start}-{window_end} len={len(window)}')
    for shift in range(0, 16):
        shifted = window[shift:]
        if shifted.startswith(b'ctf='):
            print('block', block, 'shift', shift, 'ctf= at window start')
    # test alternate bitplane offsets inside window
    for offset in range(1, 16):
        sample = window[offset:offset + 65536]
        for bit in [1, 6]:
            out = bitplane_bytes(sample, bit)
            if out.startswith(b'84'):
                print(f'block={block} offset={offset} bit={bit} starts with 0x84, len={len(out)}')
            if b'ctf=' in out[:32]:
                print(f'block={block} offset={offset} bit={bit} found ctf= in header')

block=4 window bytes 13010164-13079796 len=69632
block=8 window bytes 13010168-13079800 len=69632


## 7. Validate packet lengths and check alternate alignments
Compare declared packet lengths against available bytes and determine whether the truncation is due to an incomplete extraction window.

In [18]:
def pgp_packet_length(header: bytes) -> int:
    if len(header) < 2:
        return -1
    tag = header[0] >> 2
    new_format = bool(header[0] & 0x40)
    if new_format:
        length_type = header[1]
        if length_type < 192:
            return length_type
        if 192 <= length_type <= 223 and len(header) >= 3:
            return ((length_type - 192) << 8) + header[2] + 192
        if length_type == 255 and len(header) >= 6:
            return struct.unpack('>I', header[2:6])[0]
        return -1
    else:
        length_type = header[1] & 0x03
        if length_type == 0:
            return header[1] >> 2
        if length_type == 1 and len(header) >= 3:
            return struct.unpack('>H', header[2:4])[0]
        if length_type == 2 and len(header) >= 5:
            return struct.unpack('>I', header[2:6])[0]
        return -1

if os.path.exists('bitplane_4_1.bin'):
    data = open('bitplane_4_1.bin', 'rb').read()
    length = pgp_packet_length(data)
    print('bitplane_4_1.bin reported packet length', length, 'available', len(data))
else:
    print('bitplane_4_1.bin not found')

bitplane_4_1.bin reported packet length 560437337 available 8192


In [19]:

# Inspect existing candidate bitplane files from the workspace
import glob
candidates = sorted(glob.glob('bitplane_*.bin'))
print('bitplane files:', candidates)
for path in candidates:
    data = open(path, 'rb').read()
    print(path, 'len', len(data), 'head', data[:16])


bitplane files: ['bitplane_4_1.bin', 'bitplane_4_1_alg_1.bin', 'bitplane_4_1_alg_100.bin', 'bitplane_4_1_alg_101.bin', 'bitplane_4_1_alg_102.bin', 'bitplane_4_1_alg_103.bin', 'bitplane_4_1_alg_104.bin', 'bitplane_4_1_alg_105.bin', 'bitplane_4_1_alg_106.bin', 'bitplane_4_1_alg_107.bin', 'bitplane_4_1_alg_108.bin', 'bitplane_4_1_alg_109.bin', 'bitplane_4_1_alg_110.bin', 'bitplane_4_1_alg_116.bin', 'bitplane_4_1_alg_16.bin', 'bitplane_4_1_alg_17.bin', 'bitplane_4_1_alg_18.bin', 'bitplane_4_1_alg_19.bin', 'bitplane_4_1_alg_2.bin', 'bitplane_4_1_alg_20.bin', 'bitplane_4_1_alg_3.bin', 'bitplane_4_1_fix.bin', 'bitplane_4_1_full.bin', 'bitplane_4_6.bin', 'bitplane_4_6_full.bin', 'bitplane_8_1.bin', 'bitplane_8_6.bin']
bitplane_4_1.bin len 8192 head b'\x95Z!g\x98Y\x02\x0ct\xe4\x05=3v:D'
bitplane_4_1_alg_1.bin len 8192 head b'\x95Z!\x04\x98Y\x02\x0c\x01\xe4\x05=3v:D'
bitplane_4_1_alg_100.bin len 8192 head b'\x95Z!\x04\x98Y\x02\x0cd\xe4\x05=3v:D'
bitplane_4_1_alg_101.bin len 8192 head b'\x95Z!\x0

In [20]:
# Inspect the previously computed "full" bitplane files for PGP-like structure
for path in ['bitplane_4_1_full.bin', 'bitplane_4_6_full.bin']:
    if not os.path.exists(path):
        print('missing', path)
        continue
    data = open(path, 'rb').read()
    print('\n', path, 'len', len(data))
    print('head', data[:32])
    for i in range(min(32, len(data))):
        print(f'{i:02d}: {data[i]:02x}', end='  ')
    print()  
    # Search for plausible packet tag bytes in the first 256 bytes
    candidates = [i for i, b in enumerate(data[:256]) if b in (0x84, 0xc0, 0xd0, 0x8c, 0x94, 0x90, 0x96)]
    print('potential packet bytes in first 256:', candidates[:20])


 bitplane_4_1_full.bin len 11617
head b'\x95Z!g\x98Y\x02\x0ct\xe4\x05=3v:DtD/a6C&\x00\x05\x10s\x0673A\xa8'
00: 95  01: 5a  02: 21  03: 67  04: 98  05: 59  06: 02  07: 0c  08: 74  09: e4  10: 05  11: 3d  12: 33  13: 76  14: 3a  15: 44  16: 74  17: 44  18: 2f  19: 61  20: 36  21: 43  22: 26  23: 00  24: 05  25: 10  26: 73  27: 06  28: 37  29: 33  30: 41  31: a8  
potential packet bytes in first 256: [124, 179]

 bitplane_4_6_full.bin len 11617
head b'7S#0S2q13R6vQ6sBd\x07a0\x02\x06ERpR\x16#!p\x15\x06'
00: 37  01: 53  02: 23  03: 30  04: 53  05: 32  06: 71  07: 31  08: 33  09: 52  10: 36  11: 76  12: 51  13: 36  14: 73  15: 42  16: 64  17: 07  18: 61  19: 30  20: 02  21: 06  22: 45  23: 52  24: 70  25: 52  26: 16  27: 23  28: 21  29: 70  30: 15  31: 06  
potential packet bytes in first 256: []


In [21]:
# Current payload structure inspection
import glob, struct

candidates = sorted(glob.glob('bitplane_*.bin'))
print('candidate files:', candidates)

for name in ['bitplane_4_1.bin','bitplane_4_1_full.bin','bitplane_4_6.bin','bitplane_4_6_full.bin','bitplane_8_1.bin','bitplane_8_6.bin']:
    if not os.path.exists(name):
        continue
    data = open(name, 'rb').read()
    print('\nFILE', name, 'len', len(data))
    print('head ascii:', ''.join(chr(b) if 32<=b<127 else '.' for b in data[:32]))
    print('head hex:', data[:16].hex())
    if name.startswith('bitplane_4_1') and len(data)>=6:
        tag = data[0] >> 2
        length_type = data[1] & 0x03
        print(' packet tag', tag, 'length_type', length_type, 'byte1', data[1])
        if length_type == 2:
            print(' old packet length', struct.unpack('>I', data[2:6])[0])
    runs=[]
    cur=b''
    for b in data:
        if 32<=b<127:
            cur += bytes([b])
        else:
            if len(cur) >= 12:
                runs.append(cur)
            cur=b''
    if len(cur) >= 12:
        runs.append(cur)
    print(' printable runs (len>=12):', [r.decode('ascii',errors='ignore') for r in runs[:5]])

for name in ['bitplane_4_1_full.bin','bitplane_4_6_full.bin']:
    if not os.path.exists(name):
        continue
    data = open(name, 'rb').read()
    hits=[]
    for i,b in enumerate(data[:512]):
        if b in (0x84,0x85,0x8c,0x94,0x90,0x96,0xa4,0xb4,0xc4,0xd4):
            hits.append((i,b))
    print('\n', name, 'packet-tag-like hits first 20:', hits[:20])

import torch
obj = torch.load('ode.pt', map_location='cpu', weights_only=False)
raw6 = obj['model']['transformer.h.6.mlp.c_proj.weight'].numpy().tobytes()
raw7 = obj['model']['transformer.h.7.mlp.c_proj.weight'].numpy().tobytes()
for block in [4,8]:
    blob = bytearray()
    for i in range(0, min(len(raw6), len(raw7)), block):
        blob.extend(raw6[i:i+block])
        blob.extend(raw7[i:i+block])
    blob = bytes(blob)
    idx = blob.find(b'ctf=')
    print('\nblock', block, 'ctf idx', idx, 'blob len', len(blob))
    if idx == -1:
        continue
    snippet = blob[idx:idx+128]
    print(' snippet ascii:', ''.join(chr(b) if 32<=b<127 else '.' for b in snippet))
    print(' snippet hex:', snippet.hex())
    for pat in [b'PK', b'\x1f\x8b', b'ctf=', b'flag{', b'-----BEGIN']:
        pos = blob.find(pat, max(0, idx-2048), idx+2048)
        print('  pat', pat, 'pos', pos)


candidate files: ['bitplane_4_1.bin', 'bitplane_4_1_alg_1.bin', 'bitplane_4_1_alg_100.bin', 'bitplane_4_1_alg_101.bin', 'bitplane_4_1_alg_102.bin', 'bitplane_4_1_alg_103.bin', 'bitplane_4_1_alg_104.bin', 'bitplane_4_1_alg_105.bin', 'bitplane_4_1_alg_106.bin', 'bitplane_4_1_alg_107.bin', 'bitplane_4_1_alg_108.bin', 'bitplane_4_1_alg_109.bin', 'bitplane_4_1_alg_110.bin', 'bitplane_4_1_alg_116.bin', 'bitplane_4_1_alg_16.bin', 'bitplane_4_1_alg_17.bin', 'bitplane_4_1_alg_18.bin', 'bitplane_4_1_alg_19.bin', 'bitplane_4_1_alg_2.bin', 'bitplane_4_1_alg_20.bin', 'bitplane_4_1_alg_3.bin', 'bitplane_4_1_fix.bin', 'bitplane_4_1_full.bin', 'bitplane_4_6.bin', 'bitplane_4_6_full.bin', 'bitplane_8_1.bin', 'bitplane_8_6.bin']

FILE bitplane_4_1.bin len 8192
head ascii: .Z!g.Y..t..=3v:DtD/a6C&...s.73A.
head hex: 955a21679859020c74e4053d33763a44
 packet tag 37 length_type 2 byte1 90
 old packet length 560437337
 printable runs (len>=12): ['=3v:DtD/a6C&', 'u8VDq#`w\\&*afq', 'bqaFVKvf~@a5EV', 'gQp4&t`S#A

In [22]:
# Inspect algorithm-derived variant files for PGP-like headers or clearer structure
import glob, struct
variants = sorted(glob.glob('bitplane_4_1_alg_*.bin')) + ['bitplane_4_1_fix.bin']
print('variants count', len(variants))
for name in variants:
    if not os.path.exists(name):
        continue
    data = open(name, 'rb').read()
    head = data[:16]
    if len(data) < 16:
        continue
    tag = head[0] >> 2
    lt = head[1] & 0x03
    marker = '*' if head[0] in (0x84,0x85,0x8c,0x94,0x90,0x96,0xa4,0xb4,0xc4,0xd4,0x95) else ' '
    print(marker, name, len(data), head.hex(), 'ascii', ''.join(chr(b) if 32<=b<127 else '.' for b in head[:16]), 'tag', tag, 'lt', lt)
    if lt == 2 and len(data) >= 6:
        print('   old len', struct.unpack('>I', data[2:6])[0])
    run = b''
    for b in data:
        if 32 <= b < 127:
            run += bytes([b])
        else:
            if len(run) >= 12:
                print('   printable run', run[:64].decode('ascii', errors='ignore'))
                break
            run = b''


variants count 21
* bitplane_4_1_alg_1.bin 8192 955a21049859020c01e4053d33763a44 ascii .Z!..Y.....=3v:D tag 37 lt 2
   old len 553949273
   printable run =3v:DtD/a6C&
* bitplane_4_1_alg_100.bin 8192 955a21049859020c64e4053d33763a44 ascii .Z!..Y..d..=3v:D tag 37 lt 2
   old len 553949273
   printable run =3v:DtD/a6C&
* bitplane_4_1_alg_101.bin 8192 955a21049859020c65e4053d33763a44 ascii .Z!..Y..e..=3v:D tag 37 lt 2
   old len 553949273
   printable run =3v:DtD/a6C&
* bitplane_4_1_alg_102.bin 8192 955a21049859020c66e4053d33763a44 ascii .Z!..Y..f..=3v:D tag 37 lt 2
   old len 553949273
   printable run =3v:DtD/a6C&
* bitplane_4_1_alg_103.bin 8192 955a21049859020c67e4053d33763a44 ascii .Z!..Y..g..=3v:D tag 37 lt 2
   old len 553949273
   printable run =3v:DtD/a6C&
* bitplane_4_1_alg_104.bin 8192 955a21049859020c68e4053d33763a44 ascii .Z!..Y..h..=3v:D tag 37 lt 2
   old len 553949273
   printable run =3v:DtD/a6C&
* bitplane_4_1_alg_105.bin 8192 955a21049859020c69e4053d33763a44 ascii .Z!..Y.

In [23]:
# Search candidate bitplane files for embedded markers
import glob
markers = [b'ctf=', b'ctf{', b'flag{', b'PK', b'\x1f\x8b']
for name in sorted(glob.glob('bitplane_*.bin')):
    data = open(name,'rb').read()
    found = []
    for m in markers:
        idx = data.find(m)
        if idx != -1:
            found.append((m.decode('ascii',errors='ignore'), idx))
    if found:
        print(name, 'found', found)


In [24]:
# Systematic bitplane and transform search for the deinterleaved payload
import struct


def bitplane_bytes(data: bytes, bit: int, reverse_bits: bool = False) -> bytes:
    bits = [((c >> bit) & 1) for c in data]
    out = bytearray()
    for i in range(0, len(bits) - (len(bits) % 8), 8):
        byte = 0
        for j in range(8):
            byte |= bits[i + j] << (7 - j if reverse_bits else j)
        out.append(byte)
    return bytes(out)


def nibble_swap(data: bytes) -> bytes:
    return bytes(((b >> 4) | ((b & 0xF) << 4)) for b in data)


def signature_info(data: bytes) -> str:
    sigs = [b'ctf=', b'ctf{', b'flag{', b'PK', b'\x1f\x8b', b'-----BEGIN', b'%PDF', b'RIFF', b'7z', b'BZh']
    found = []
    for s in sigs:
        idx = data.find(s)
        if idx != -1:
            found.append(f"{s.decode('ascii',errors='ignore')}@{idx}")
    return ', '.join(found)


def make_deinterleave(raw6, raw7, block):
    out = bytearray()
    for i in range(0, min(len(raw6), len(raw7)), block):
        out.extend(raw6[i:i+block])
        out.extend(raw7[i:i+block])
    return bytes(out)

obj = torch.load('ode.pt', map_location='cpu', weights_only=False)
raw6 = obj['model']['transformer.h.6.mlp.c_proj.weight'].numpy().tobytes()
raw7 = obj['model']['transformer.h.7.mlp.c_proj.weight'].numpy().tobytes()

for block in [4, 8]:
    blob = make_deinterleave(raw6, raw7, block)
    idx = blob.find(b'ctf=')
    print(f'block={block}, ctf idx={idx}, blob len={len(blob)}')
    if idx == -1:
        continue
    sample = blob[max(0, idx - 4096): idx + 65536]
    print('  sample len', len(sample))
    print('  sample sigs', signature_info(sample[:256]))
    for bit in range(8):
        out_plain = bitplane_bytes(sample, bit)
        out_rev = bitplane_bytes(sample, bit, reverse_bits=True)
        for name, out in [('plain', out_plain), ('revbit', out_rev)]:
            sig = signature_info(out[:256])
            if sig:
                print(f'    bit={bit} {name} sigs={sig} printable={sum(32<=c<127 for c in out[:256])}/256')
        if bit in (1, 6):
            nplain = nibble_swap(out_plain[:1024])
            nrev = nibble_swap(out_rev[:1024])
            if signature_info(nplain):
                print(f'    bit={bit} plain+nibble sig={signature_info(nplain)}')
            if signature_info(nrev):
                print(f'    bit={bit} revbit+nibble sig={signature_info(nrev)}')
    print('  first 32 plain bytes', out_plain[:32].hex())
    print('  first 32 revbit bytes', out_rev[:32].hex())

print('\nCandidate file header overview:')
for fn in ['bitplane_4_1_full.bin', 'bitplane_4_6_full.bin']:
    if not os.path.exists(fn):
        continue
    data = open(fn, 'rb').read()
    print(fn, 'len', len(data), 'head:', data[:16].hex(), 'ascii:', ''.join(chr(b) if 32<=b<127 else '.' for b in data[:16]))
    print('  signature', signature_info(data[:256]))
    if len(data) >= 6:
        t = data[0] >> 2
        lt = data[1] & 0x03
        packet_len = struct.unpack('>I', data[2:6])[0] if lt == 2 else None
        print('  pkt tag', t, 'len_type', lt, 'byte1', data[1], 'packet len', packet_len)


block=4, ctf idx=13014260, blob len=13107200
  sample len 69632
  sample sigs 
  first 32 plain bytes 17068d943c4092d4e38f120c55ababc0a017ba86637e76055604b5d79d759a9b
  first 32 revbit bytes e860b1293c02492bc7f14830aad5d50305e85d61c67e6ea06a20adebb9ae59d9
block=8, ctf idx=13014264, blob len=13107200
  sample len 69632
  sample sigs 
  first 32 plain bytes 67804d390c9442edf318c250b5aa0bac70b16a68e377565046b0759d5d97bae9
  first 32 revbit bytes e601b29c302942b7cf18430aad55d0350e8d5616c7ee6a0a620daeb9bae95d97

Candidate file header overview:
bitplane_4_1_full.bin len 11617 head: 955a21679859020c74e4053d33763a44 ascii: .Z!g.Y..t..=3v:D
  signature 
  pkt tag 37 len_type 2 byte1 90 packet len 560437337
bitplane_4_6_full.bin len 11617 head: 37532330533271313352367651367342 ascii: 7S#0S2q13R6vQ6sB
  signature 
  pkt tag 13 len_type 3 byte1 83 packet len None


In [25]:
# Deeper analysis of candidate payload files and header transforms
from collections import Counter

candidates = ['bitplane_4_1.bin','bitplane_4_1_full.bin','bitplane_4_6.bin','bitplane_4_6_full.bin','bitplane_8_1.bin','bitplane_8_6.bin']


def reverse_bit_order(data: bytes) -> bytes:
    table = bytes(int('{:08b}'.format(b)[::-1], 2) for b in range(256))
    return data.translate(table)


def printable_stats(data: bytes, name=''):
    printable = sum(32 <= b < 127 for b in data)
    print(f'{name} printable {printable}/{len(data)} ratio {printable/len(data):.4f}')
    runs = []
    cur = b''
    for b in data:
        if 32 <= b < 127:
            cur += bytes([b])
        else:
            if len(cur) >= 8:
                runs.append(cur)
            cur = b''
    if len(cur) >= 8:
        runs.append(cur)
    print('  runs first 5', [r.decode('ascii', errors='ignore') for r in runs[:5]])
    return runs

for fn in candidates:
    if not os.path.exists(fn):
        continue
    data = open(fn, 'rb').read()
    print('\n===', fn, 'len', len(data))
    printable_stats(data[:256], name='header')
    print(' header hex', data[:32].hex())
    print(' header ascii', ''.join(chr(b) if 32<=b<127 else '.' for b in data[:32]))
    for transformed, label in [
        (data, 'plain'),
        (reverse_bit_order(data), 'revbits'),
        (data[::-1], 'revbytes'),
        (nibble_swap(data), 'nibbleswap'),
        (reverse_bit_order(nibble_swap(data)), 'nibbleswap+revbits'),
    ]:
        sig = signature_info(transformed[:256])
        print(' ', label, 'sig', sig or 'none', 'head hex', transformed[:16].hex())
    print(' whole signature', signature_info(data))



=== bitplane_4_1.bin len 8192
header printable 159/256 ratio 0.6211
  runs first 5 ['=3v:DtD/a6C&', 'edAeCesR', '3r;w66Tw', '61vkj1P%']
 header hex 955a21679859020c74e4053d33763a4474442f613643260005107306373341a8
 header ascii .Z!g.Y..t..=3v:DtD/a6C&...s.73A.
  plain sig none head hex 955a21679859020c74e4053d33763a44
  revbits sig none head hex a95a84e6199a40302e27a0bccc6e5c22
  revbytes sig none head hex 63673344b41317430316666d756f1392
  nibbleswap sig none head hex 59a51276899520c0474e50d33367a344
  nibbleswap+revbits sig none head hex 9aa5486e91a90403e2720acbcce6c522
 whole signature 7z@5965

=== bitplane_4_1_full.bin len 11617
header printable 159/256 ratio 0.6211
  runs first 5 ['=3v:DtD/a6C&', 'edAeCesR', '3r;w66Tw', '61vkj1P%']
 header hex 955a21679859020c74e4053d33763a4474442f613643260005107306373341a8
 header ascii .Z!g.Y..t..=3v:DtD/a6C&...s.73A.
  plain sig none head hex 955a21679859020c74e4053d33763a44
  revbits sig none head hex a95a84e6199a40302e27a0bccc6e5c22
  revbyte

In [26]:
# Inspect the 7z-like signature in the bitplane_4_1 payloads
needle = b'7z\xbc\xaf\x27\x1c'
for fn in ['bitplane_4_1.bin', 'bitplane_4_1_full.bin']:
    if not os.path.exists(fn):
        continue
    data = open(fn, 'rb').read()
    idx = data.find(needle)
    print('\n===', fn, 'len', len(data), '7z idx', idx)
    if idx != -1:
        snippet = data[idx:idx+64]
        print('header hex', snippet[:16].hex())
        print('ascii', ''.join(chr(b) if 32<=b<127 else '.' for b in snippet))
        print('next 128 bytes', data[idx:idx+128].hex())
    else:
        matches = [i for i in range(len(data)-1) if data[i:i+2]==b'7z']
        print('7z partial positions', matches[:20])
        if matches:
            for m in matches[:5]:
                snippet = data[m:m+32]
                print(' pos', m, 'hex', snippet.hex(), 'ascii', ''.join(chr(b) if 32<=b<127 else '.' for b in snippet))



=== bitplane_4_1.bin len 8192 7z idx -1
7z partial positions [5965]
 pos 5965 hex 377a12103a714544845a11289d31032bc2c762157733670de210762244200202 ascii 7z..:qED.Z.(.1.+..b.w3g...v"D ..

=== bitplane_4_1_full.bin len 11617 7z idx -1
7z partial positions [5965]
 pos 5965 hex 377a12103a714544845a11289d31032bc2c762157733670de210762244200202 ascii 7z..:qED.Z.(.1.+..b.w3g...v"D ..


In [27]:
# Test simple XOR/key and bit-rotation transforms on the deinterleaved ctf region

def rotate_left(b, n):
    return ((b << n) & 0xFF) | (b >> (8 - n))

def rotate_right(b, n):
    return ((b >> n) | ((b << (8 - n)) & 0xFF)) & 0xFF

def printable_score(data: bytes) -> float:
    return sum(32 <= b < 127 for b in data) / len(data)

for block in [4, 8]:
    blob = deinterleave(raw6, raw7, block)
    idx = blob.find(b'ctf=')
    print(f'\nblock {block} ctf idx {idx}')
    if idx == -1:
        continue
    data = blob[idx:idx+256]
    scores = []
    for key in range(1, 256):
        x = bytes(b ^ key for b in data)
        score = printable_score(x)
        if score > 0.75:
            signatures = signature_info(x[:64])
            scores.append((score, key, signatures))
    scores.sort(reverse=True)
    print(' top XOR keys:', scores[:10])

    rot_scores = []
    for n in range(1, 8):
        x = bytes(rotate_left(b, n) for b in data)
        score = printable_score(x)
        if score > 0.6:
            rot_scores.append(('l', n, score, signature_info(x[:64])))
        x2 = bytes(rotate_right(b, n) for b in data)
        score2 = printable_score(x2)
        if score2 > 0.6:
            rot_scores.append(('r', n, score2, signature_info(x2[:64])))
    print(' rotation hits:', rot_scores)

    swapped = nibble_swap(data)
    print(' nibble swap score', printable_score(swapped), 'sig', signature_info(swapped[:64]))
    revbits = bytes(int('{:08b}'.format(b)[::-1], 2) for b in data)
    print(' revbits score', printable_score(revbits), 'sig', signature_info(revbits[:64]))
    revbytes = data[::-1]
    print(' revbytes score', printable_score(revbytes), 'sig', signature_info(revbytes[:64]))



block 4 ctf idx 13014260
 top XOR keys: []
 rotation hits: []
 nibble swap score 0.35546875 sig 
 revbits score 0.41796875 sig 
 revbytes score 0.3671875 sig 

block 8 ctf idx 13014264
 top XOR keys: []
 rotation hits: []
 nibble swap score 0.35546875 sig 
 revbits score 0.41796875 sig 
 revbytes score 0.375 sig 


In [28]:
# Test single-byte XOR detection on candidate full bitplane files

candidates = ['bitplane_4_1_full.bin', 'bitplane_4_6_full.bin']

def score_plaintext(data: bytes) -> float:
    return sum(32 <= b < 127 for b in data) / len(data)

for fn in candidates:
    if not os.path.exists(fn):
        continue
    data = open(fn, 'rb').read()
    print('\n===', fn, 'len', len(data))
    best = []
    sample = data[:4096]
    for key in range(256):
        x = bytes(b ^ key for b in sample)
        score = score_plaintext(x)
        best.append((score, key))
    best.sort(reverse=True)
    for score, key in best[:10]:
        x = bytes(b ^ key for b in sample[:64])
        print(' key', key, 'score', score, 'head', x[:64])



=== bitplane_4_1_full.bin len 11617
 key 51 score 0.640625 head b'\xa6i\x12T\xabj1?G\xd76\x0e\x00E\twGw\x1cR\x05p\x1536#@5\x04\x00r\x9bp\xb2VWrVpV@a\x91&\x15.\xdcFb\xdd\x10\x85s\x01C$g*\x97\x03US\x02('
 key 36 score 0.640625 head b'\xb1~\x05C\xbc}&(P\xc0!\x19\x17R\x1e`P`\x0bE\x12g\x02$!4W"\x13\x17e\x8cg\xa5A@eAgAWv\x861\x029\xcbQu\xca\x07\x92d\x16T3p=\x80\x14BD\x15?'
 key 33 score 0.640625 head b"\xb4{\x00F\xb9x#-U\xc5$\x1c\x12W\x1beUe\x0e@\x17b\x07!$1R'\x16\x12`\x89b\xa0DE`DbDRs\x834\x07<\xceTp\xcf\x02\x97a\x13Q6u8\x85\x11GA\x10:"
 key 54 score 0.640380859375 head b'\xa3l\x17Q\xaeo4:B\xd23\x0b\x05@\x0crBr\x19W\x00u\x1063&E0\x01\x05w\x9eu\xb7SRwSuSEd\x94#\x10+\xd9Cg\xd8\x15\x80v\x04F!b/\x92\x06PV\x07-'
 key 48 score 0.640380859375 head b"\xa5j\x11W\xa8i2<D\xd45\r\x03F\ntDt\x1fQ\x06s\x1605 C6\x07\x03q\x98s\xb1UTqUsUCb\x92%\x16-\xdfEa\xde\x13\x86p\x02@'d)\x94\x00VP\x01+"
 key 32 score 0.640380859375 head b'\xb5z\x01G\xb8y",T\xc4%\x1d\x13V\x1adTd\x0fA\x16c\x06 %0S&\x17\x13a\x88c\xa1EDaEc

In [29]:
# Search for known markers after single-byte XOR on candidate files
markers = [b'ctf=', b'ctf{', b'flag{', b'PK', b'7z\xbc\xaf\x27\x1c', b'\x1f\x8b', b'-----BEGIN']
for fn in ['bitplane_4_1_full.bin', 'bitplane_4_6_full.bin']:
    if not os.path.exists(fn):
        continue
    data = open(fn, 'rb').read()
    print('\n===', fn)
    for key in range(256):
        x = bytes(b ^ key for b in data)
        found = []
        for m in markers:
            idx = x.find(m)
            if idx != -1:
                found.append((m.decode('ascii', errors='ignore'), idx))
        if found:
            print(' key', key, 'found', found)
            print(' head', x[:64])
            break



=== bitplane_4_1_full.bin
 key 1 found [('PK', 7872)]
 head b'\x94[ f\x99X\x03\ru\xe5\x04<2w;EuE.`7B\'\x01\x04\x11r\x0762@\xa9B\x80de@dBdrS\xa3\x14\'\x1c\xeetP\xef"\xb7A3q\x16U\x18\xa51ga0\x1a'

=== bitplane_4_6_full.bin


In [30]:
# Inspect the bitplane_4_1_full.bin payload after XOR key 1
fn = 'bitplane_4_1_full.bin'
data = open(fn, 'rb').read()
key = 1
x = bytes(b ^ key for b in data)
for target in [b'PK\x03\x04', b'PK\x05\x06', b'PK\x07\x08']:
    idx = x.find(target)
    print('target', target, 'idx', idx)
    if idx != -1:
        print(' snippet', x[idx:idx+64])
        print(' hex', x[idx:idx+32].hex())
        print(' ascii', ''.join(chr(b) if 32<=b<127 else '.' for b in x[idx:idx+32]))

# Try loading as a zip archive if possible
import zipfile
from io import BytesIO

try:
    zf = zipfile.ZipFile(BytesIO(x))
    print('\nzipfile list:')
    print(zf.namelist())
    for name in zf.namelist():
        print(' name', name, 'size', zf.getinfo(name).file_size)
except Exception as e:
    print('\nzipfile failed:', repr(e))

# Write the XOR'd payload for manual inspection if needed
with open('bitplane_4_1_full_xor1.bin', 'wb') as f:
    f.write(x)
print('wrote bitplane_4_1_full_xor1.bin')


target b'PK\x03\x04' idx -1
target b'PK\x05\x06' idx -1
target b'PK\x07\x08' idx -1

zipfile failed: BadZipFile('File is not a zip file')
wrote bitplane_4_1_full_xor1.bin


In [31]:
# Examine the XOR-1 transformed output around the PK hit and search for other file signatures
fn = 'bitplane_4_1_full_xor1.bin'
data = open(fn, 'rb').read()
for s in [b'PK\x03\x04', b'PK\x05\x06', b'PK\x07\x08', b'7z\xbc\xaf\x27\x1c', b'\x1f\x8b', b'%PDF', b'Rar!']:
    print('search', s, 'idx', data.find(s))

# print the PK context found previously at 7872
idx = data.find(b'PK')
print('first PK idx', idx)
if idx != -1:
    snippet = data[idx:idx+64]
    print('hex', snippet.hex())
    print('ascii', ''.join(chr(b) if 32<=b<127 else '.' for b in snippet))

# search for printable run around the PK position
if idx != -1:
    window = data[idx-64:idx+192]
    print('window printable ratio', sum(32<=b<127 for b in window)/len(window))
    print('window ascii', ''.join(chr(b) if 32<=b<127 else '.' for b in window))
    print('window hex', window.hex())


search b'PK\x03\x04' idx -1
search b'PK\x05\x06' idx -1
search b'PK\x07\x08' idx -1
search b"7z\xbc\xaf'\x1c" idx -1
search b'\x1f\x8b' idx -1
search b'%PDF' idx -1
search b'Rar!' idx -1
first PK idx 7872
hex 504b61354f03027c37363d1e275727d334d0252135226003736700d3d2555b144c92401616eb70fb37542335217706777713331f17204e2b633379447230dc32
ascii PKa5O..|76=.'W'.4.%!5"`.sg...U[.L.@...p.7T#5!w.ww.3.. N+c3yDr0.2
window printable ratio 0.59375
window ascii .As@Q$u$R.F....e...uG.A#cG r.F.bDl.e7D....#..."@d2!3.c.hV.7febG.PKa5O..|76=.'W'.4.%!5"`.sg...U[.L.@...p.7T#5!w.ww.3.. N+c3yDr0.2..a.$FS!.6u.&..6..FG@T3#B"u.r..R$.6p#.1!"3.6T#.w.A....d.e.....s. 0..egb..#R@wC.V..Q-Me.P..V..%!.. .w6r..4...t.1....J ......CG...
window hex 1e4173405124752452c34603b3b2026512a9127547ae41236347207212461562446c856537441b0f079123c3030822406432213382632e6856053766656247d2504b61354f03027c37363d1e275727d334d0252135226003736700d3d2555b144c92401616eb70fb37542335217706777713331f17204e2b633379447230dc3203f76187244653210236750

In [32]:
# Score XOR keys for likely English text in candidate payloads
import math
english_chars = set(b' etaoinshrdlcumwfgypbvkjxqzETAOINSHRDLCUMWFGYPBVKJXQZ')

def english_score(data: bytes) -> float:
    printable = sum(32 <= b < 127 for b in data)
    spaces = data.count(0x20)
    common = sum(b in english_chars for b in data)
    return printable/len(data) + 2 * spaces/len(data) + common/len(data)

for fn in ['bitplane_4_6_full.bin', 'bitplane_4_1_full.bin']:
    if not os.path.exists(fn):
        continue
    data = open(fn, 'rb').read()[:8192]
    scores = []
    for key in range(256):
        x = bytes(b ^ key for b in data)
        score = english_score(x)
        scores.append((score, key))
    scores.sort(reverse=True)
    print('\n===', fn)
    for score, key in scores[:10]:
        x = bytes(b ^ key for b in data[:128])
        print(' key', key, 'score', round(score,4), 'head', x[:128])



=== bitplane_4_6_full.bin
 key 69 score 1.3813 head b'r\x16fu\x16w4tv\x17s3\x14s6\x07!B$uGC\x00\x175\x17Sfd5PC@Ge\x147C\'TB QrD`\x17P\x04de"F\x12\x15vVdgt\x04%wG\x07sG\x16tDPV\x13\'S\x00\x13C&$gBPU\x13\x16\x10VeSq!QrV A2"tvr\x10qQS6Af1qu@\'\x14@dTbWEwRApu \''
 key 71 score 1.3798 head b'p\x14dw\x14u6vt\x15q1\x16q4\x05#@&wEA\x02\x157\x15Qdf7RABEg\x165A%V@"SpFb\x15R\x06fg D\x10\x17tTfev\x06\'uE\x05qE\x14vFRT\x11%Q\x02\x11A$&e@RW\x11\x14\x12TgQs#SpT"C0 vtp\x12sSQ4Cd3swB%\x16BfV`UGuPCrw"%'
 key 117 score 1.379 head b'B&VE&G\x04DF\'C\x03$C\x067\x11r\x14Ews0\'\x05\'cVT\x05`spwU$\x07s\x17dr\x10aBtP\'`4TU\x12v"%FfTWD4\x15Gw7Cw&Dt`f#\x17c0#s\x16\x14Wr`e#& fUcA\x11aBf\x10q\x02\x12DFB Aac\x06qV\x01AEp\x17$pTdRguGbq@E\x10\x17'
 key 98 score 1.3788 head b"U1AR1P\x13SQ0T\x143T\x11 \x06e\x03R`d'0\x120tAC\x12wdg`B3\x10d\x00se\x07vUcG0w#CB\x05a52QqC@S#\x02P` T`1Scwq4\x00t'4d\x01\x03@ewr417qBtV\x06vUq\x07f\x15\x05SQU7Vvt\x11fA\x16VRg\x003gCsEpbPufWR\x07\x00"
 key 118 score 1.3784 head b"A%UF%D\x07GE$@\

In [33]:
# Summarize English-scoring results with compact output
english_chars = set(b' etaoinshrdlcumwfgypbvkjxqzETAOINSHRDLCUMWFGYPBVKJXQZ')

def english_score(data: bytes) -> float:
    printable = sum(32 <= b < 127 for b in data)
    spaces = data.count(0x20)
    common = sum(b in english_chars for b in data)
    return printable/len(data) + 2 * spaces/len(data) + common/len(data)

for fn in ['bitplane_4_6_full.bin', 'bitplane_4_1_full.bin']:
    if not os.path.exists(fn):
        continue
    data = open(fn, 'rb').read()[:4096]
    scores = []
    for key in range(256):
        x = bytes(b ^ key for b in data)
        score = english_score(x)
        scores.append((score, key))
    scores.sort(reverse=True)
    print('\n===', fn)
    for score, key in scores[:10]:
        x = bytes(b ^ key for b in data[:32])
        print(' key', key, 'score', round(score,4), 'hexhead', x[:16].hex(), 'asciihead', ''.join(chr(b) if 32<=b<127 else '.' for b in x[:16]))



=== bitplane_4_6_full.bin
 key 69 score 1.3984 hexhead 72166675167734747617733314733607 asciihead r.fu.w4tv.s3.s6.
 key 70 score 1.3975 hexhead 71156576157437777514703017703504 asciihead q.ev.t7wu.p0.p5.
 key 64 score 1.3926 hexhead 77136370137231717312763611763302 asciihead w.cp.r1qs.v6.v3.
 key 86 score 1.3892 hexhead 61057566056427676504602007602514 asciihead a.uf.d'ge.` .`%.
 key 68 score 1.3884 hexhead 73176774177635757716723215723706 asciihead s.gt.v5uw.r2.r7.
 key 71 score 1.3867 hexhead 70146477147536767415713116713405 asciihead p.dw.u6vt.q1.q4.
 key 66 score 1.3862 hexhead 75116172117033737110743413743100 asciihead u.ar.p3sq.t4.t1.
 key 117 score 1.3853 hexhead 42265645264704444627430324430637 asciihead B&VE&G.DF'C.$C.7
 key 82 score 1.3848 hexhead 65017162016023636100642403642110 asciihead e.qb.`#ca.d$.d!.
 key 87 score 1.3845 hexhead 60047467046526666405612106612415 asciihead `.tg.e&fd.a!.a$.

=== bitplane_4_1_full.bin
 key 83 score 1.0645 hexhead c6097234cb0a515f27b7566e60

In [34]:
# Inspect top XOR keys for bitplane_4_6_full.bin in more detail
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
keys = [69, 70, 64, 86, 68, 71, 66, 117, 82, 87]
markers = [b'ctf=', b'flag{', b'PK', b'http', b'Augusta', b'augusta', b'arcus', b'Ode']
for key in keys:
    x = bytes(b ^ key for b in data)
    print('\n=== key', key)
    print(' first 64:', x[:64])
    print(' ascii head:', ''.join(chr(b) if 32<=b<127 else '.' for b in x[:64]))
    print(' printable ratio', sum(32<=b<127 for b in x[:256]) / 256)
    for m in markers:
        idx = x.find(m)
        if idx != -1:
            print(' marker', m, 'at', idx)



=== key 69
 first 64: b'r\x16fu\x16w4tv\x17s3\x14s6\x07!B$uGC\x00\x175\x17Sfd5PC@Ge\x147C\'TB QrD`\x17P\x04de"F\x12\x15vVdgt\x04%wG'
 ascii head: r.fu.w4tv.s3.s6.!B$uGC..5.Sfd5PC@Ge.7C'TB QrD`.P.de"F..vVdgt.%wG
 printable ratio 0.7890625

=== key 70
 first 64: b'q\x15ev\x15t7wu\x14p0\x17p5\x04"A\'vD@\x03\x146\x14Peg6S@CDf\x174@$WA#RqGc\x14S\x07gf!E\x11\x16uUgdw\x07&tD'
 ascii head: q.ev.t7wu.p0.p5."A'vD@..6.Peg6S@CDf.4@$WA#RqGc.S.gf!E..uUgdw.&tD
 printable ratio 0.7890625

=== key 64
 first 64: b'w\x13cp\x13r1qs\x12v6\x11v3\x02$G!pBF\x05\x120\x12Vca0UFEB`\x112F"QG%TwAe\x12U\x01a`\'C\x17\x10sSabq\x01 rB'
 ascii head: w.cp.r1qs.v6.v3.$G!pBF..0.Vca0UFEB`.2F"QG%TwAe.U.a`'C..sSabq. rB
 printable ratio 0.7890625

=== key 86
 first 64: b"a\x05uf\x05d'ge\x04` \x07`%\x142Q7fTP\x13\x04&\x04@uw&CPSTv\x07$P4GQ3BaWs\x04C\x17wv1U\x01\x06eEwtg\x176dT"
 ascii head: a.uf.d'ge.` .`%.2Q7fTP..&.@uw&CPSTv.$P4GQ3BaWs.C.wv1U..eEwtg.6dT
 printable ratio 0.7890625

=== key 68
 first 64: b's\x17gt\x17v5uw\x16r

In [35]:
# Check alternating byte streams after XOR decryption for the top candidate keys
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
keys = [69, 70, 64, 86, 68, 71, 66, 117, 82, 87]

for key in keys:
    x = bytes(b ^ key for b in data)
    even = x[0::2]
    odd = x[1::2]
    print('\n=== key', key)
    print(' even head', even[:64])
    print(' ascii even', ''.join(chr(b) if 32<=b<127 else '.' for b in even[:64]))
    print(' odd head', odd[:64])
    print(' ascii odd', ''.join(chr(b) if 32<=b<127 else '.' for b in odd[:64]))
    print(' even printable', sum(32<=b<127 for b in even[:256])/256, 'odd printable', sum(32<=b<127 for b in odd[:256])/256)



=== key 69
 even head b'rf\x164vs\x146!$G\x005SdP@e7\'BQD\x17\x04eF\x15Vg\x04w\x07GtP\x13S\x13&gP\x13\x10eqQVA"v\x10Q6fq@\x14dbERp '
 ascii even rf.4vs.6!$G.5SdP@e7'BQD..eF.Vg.w.GtP.S.&gP..eqQVA"v.Q6fq@.dbERp 
 odd head b'\x16uwt\x173s\x07BuC\x17\x17f5CG\x14CT r`Pd"\x12vdt%Gs\x16DV\'\x00C$BU\x16VS!r 2trqSA1u\'@TWwAu\''
 ascii odd .uwt.3s.BuC..f5CG.CT r`Pd".vdt%Gs.DV'.C$BU.VS!r 2trqSA1u'@TWwAu'
 even printable 0.765625 odd printable 0.83203125

=== key 70
 even head b'qe\x157up\x175"\'D\x036PgSCf4$ARG\x14\x07fE\x16Ud\x07t\x04DwS\x10P\x10%dS\x10\x13frRUB!u\x13R5erC\x17gaFQs#'
 ascii even qe.7up.5"'D.6PgSCf4$ARG..fE.Ud.t.DwS.P.%dS..frRUB!u.R5erC.gaFQs#
 odd head b'\x15vtw\x140p\x04Av@\x14\x14e6@D\x17@W#qcSg!\x11ugw&Dp\x15GU$\x03@\'AV\x15UP"q#1wqrPB2v$CWTtBv$'
 ascii odd .vtw.0p.Av@..e6@D.@W#qcSg!.ugw&Dp.GU$.@'AV.UP"q#1wqrPB2v$CWTtBv$
 even printable 0.765625 odd printable 0.83203125

=== key 64
 even head b'wc\x131sv\x113$!B\x050VaUE`2"GTA\x12\x01`C\x10Sb\x01r\x02BqU\x16V\x16#bU\x16\x15`

In [36]:
# Focused alternating-byte inspection for the top few XOR keys on bitplane_4_6_full.bin
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
for key in [69, 70, 64]:
    x = bytes(b ^ key for b in data)
    even = x[0::2][:32]
    odd = x[1::2][:32]
    print('\n=== key', key)
    print(' even ascii', ''.join(chr(b) if 32<=b<127 else '.' for b in even))
    print(' odd ascii', ''.join(chr(b) if 32<=b<127 else '.' for b in odd))
    print(' even hex', even.hex())
    print(' odd hex', odd.hex())



=== key 69
 even ascii rf.4vs.6!$G.5SdP@e7'BQD..eF.Vg.w
 odd ascii .uwt.3s.BuC..f5CG.CT r`Pd".vdt%G
 even hex 7266163476731436212447003553645040653727425144170465461556670477
 odd hex 1675777417337307427543171766354347144354207260506422127664742547

=== key 70
 even ascii qe.7up.5"'D.6PgSCf4$ARG..fE.Ud.t
 odd ascii .vtw.0p.Av@..e6@D.@W#qcSg!.ugw&D
 even hex 7165153775701735222744033650675343663424415247140766451655640774
 odd hex 1576747714307004417640141465364044174057237163536721117567772644

=== key 64
 even ascii wc.1sv.3$!B.0VaUE`2"GTA..`C.Sb.r
 odd ascii .prq.6v.GpF..c0FB.FQ%weUa'.saq B
 even hex 7763133173761133242142053056615545603222475441120160431053620172
 odd hex 1370727112367602477046121263304642114651257765556127177361712042


In [37]:
# Test second-stage transforms on the best XOR candidates for bitplane_4_6_full.bin
from itertools import islice

fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
keys = [69, 70]

for key in keys:
    x = bytes(b ^ key for b in data)
    print('\n=== key', key)
    for label, y in [('plain', x), ('nibbleswap', nibble_swap(x)), ('revbits', bytes(int('{:08b}'.format(b)[::-1], 2) for b in x)), ('revbytes', x[::-1])]:
        head = y[:64]
        print(label, 'ascii', ''.join(chr(b) if 32<=b<127 else '.' for b in head))
        print(label, 'hex', head.hex())
        print(label, 'marker ctf=', y.find(b'ctf='), 'flag=', y.find(b'flag{'))



=== key 69
plain ascii r.fu.w4tv.s3.s6.!B$uGC..5.Sfd5PC@Ge.7C'TB QrD`.P.de"F..vVdgt.%wG
plain hex 72166675167734747617733314733607214224754743001735175366643550434047651437432754422051724460175004646522461215765664677404257747
plain marker ctf= -1 flag= -1
nibbleswap ascii 'afWawCGgq73A7cp.$BWt4.qSq5fFS.4.tVAs4rE$..'D.q.@FV"d!QgeFvG@Rwt
nibbleswap hex 27616657617743476771373341376370122442577434007153713566465305340474564173347245240215274406710540465622642151676546764740527774
nibbleswap marker ctf= -1 flag= -1
revbits ascii Nhf.h.,.n...(.l..B$........f&......(...*B..N"... &.DbH.nj&.. ...
revbits hex 4e6866ae68ee2c2e6ee8cecc28ce6ce0844224aee2c200e8ace8ca6626ac0ac202e2a628ecc2e42a42048a4e2206e80a2026a6446248a86e6a26e62e20a4eee2
revbits marker ctf= -1 flag= -1
revbytes ascii at.gP'.vbbSRuu..6D.T#V$%7e.uE..#4 7 f`q6as4uFuS Ba%S.d5R5Gv.RWT.
revbytes hex 61741567502716766262535275751205364406542356242537651475451001233420372066607136617334754675532042612553126435523547761052575414
revbyte

In [38]:
# Check the symbol alphabet size for the top XOR candidate decryptions
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
for key in [69, 70, 64, 86, 68, 71, 66, 117, 82, 87]:
    x = bytes(b ^ key for b in data[:8192])
    unique = sorted(set(x))
    print('\n=== key', key, 'unique count', len(unique), 'unique chars')
    print(' ', ''.join(chr(b) if 32<=b<127 else '.' for b in unique[:50]))
    counts = [(unique.count(u), u) for u in unique] if len(unique) <= 32 else []
    if counts:
        print(' counts (sample)', counts[:20])



=== key 69 unique count 64 unique chars
  ................ !"#$%&'01234567@ABCDEFGPQRSTUVW`a

=== key 70 unique count 64 unique chars
  ................ !"#$%&'01234567@ABCDEFGPQRSTUVW`a

=== key 64 unique count 64 unique chars
  ................ !"#$%&'01234567@ABCDEFGPQRSTUVW`a

=== key 86 unique count 64 unique chars
  ................ !"#$%&'01234567@ABCDEFGPQRSTUVW`a

=== key 68 unique count 64 unique chars
  ................ !"#$%&'01234567@ABCDEFGPQRSTUVW`a

=== key 71 unique count 64 unique chars
  ................ !"#$%&'01234567@ABCDEFGPQRSTUVW`a

=== key 66 unique count 64 unique chars
  ................ !"#$%&'01234567@ABCDEFGPQRSTUVW`a

=== key 117 unique count 64 unique chars
  ................ !"#$%&'01234567@ABCDEFGPQRSTUVW`a

=== key 82 unique count 64 unique chars
  ................ !"#$%&'01234567@ABCDEFGPQRSTUVW`a

=== key 87 unique count 64 unique chars
  ................ !"#$%&'01234567@ABCDEFGPQRSTUVW`a


In [39]:
# Determine the exact 64-symbol alphabet for the best XOR candidate
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
for key in [69, 70, 64, 86, 68, 71, 66, 117, 82, 87]:
    x = bytes(b ^ key for b in data[:8192])
    unique = sorted(set(x))
    print('\n=== key', key)
    print('min/max', min(unique), max(unique))
    print('unique count', len(unique))
    print('chars', ''.join(chr(b) for b in unique))



=== key 69
min/max 0 119
unique count 64
chars   !"#$%&'01234567@ABCDEFGPQRSTUVW`abcdefgpqrstuvw

=== key 70
min/max 0 119
unique count 64
chars   !"#$%&'01234567@ABCDEFGPQRSTUVW`abcdefgpqrstuvw

=== key 64
min/max 0 119
unique count 64
chars   !"#$%&'01234567@ABCDEFGPQRSTUVW`abcdefgpqrstuvw

=== key 86
min/max 0 119
unique count 64
chars   !"#$%&'01234567@ABCDEFGPQRSTUVW`abcdefgpqrstuvw

=== key 68
min/max 0 119
unique count 64
chars   !"#$%&'01234567@ABCDEFGPQRSTUVW`abcdefgpqrstuvw

=== key 71
min/max 0 119
unique count 64
chars   !"#$%&'01234567@ABCDEFGPQRSTUVW`abcdefgpqrstuvw

=== key 66
min/max 0 119
unique count 64
chars   !"#$%&'01234567@ABCDEFGPQRSTUVW`abcdefgpqrstuvw

=== key 117
min/max 0 119
unique count 64
chars   !"#$%&'01234567@ABCDEFGPQRSTUVW`abcdefgpqrstuvw

=== key 82
min/max 0 119
unique count 64
chars   !"#$%&'01234567@ABCDEFGPQRSTUV

In [40]:
# Decode the 64-symbol stream into binary using the 6-bit octal-style mapping

def symbol_from_byte(b: int) -> int:
    return ((b >> 4) & 0x7) * 8 + (b & 0x7)

def decode_6bit_stream(encoded: bytes) -> bytes:
    symbols = [symbol_from_byte(b) for b in encoded]
    out = bytearray()
    for i in range(0, len(symbols) - (len(symbols) % 4), 4):
        v = (symbols[i] << 18) | (symbols[i+1] << 12) | (symbols[i+2] << 6) | symbols[i+3]
        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
    return bytes(out)

fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
for key in [69, 70, 64, 86, 68, 71, 66, 117, 82, 87]:
    x = bytes(b ^ key for b in data)
    decoded = decode_6bit_stream(x)
    print('\n=== key', key, 'decoded len', len(decoded))
    print('head hex', decoded[:32].hex())
    print('head ascii', ''.join(chr(b) if 32<=b<127 else '.' for b in decoded[:32]))
    print('magic search PK', decoded.find(b'PK'), 'gzip', decoded.find(b'\x1f\x8b'), '7z', decoded.find(b'7z\xbc\xaf\x27\x1c'), 'ctf=', decoded.find(b'ctf='), 'flag{', decoded.find(b'flag{'))



=== key 69 decoded len 8712
head hex e8edbd3bf73cf8fedb33b78746253d9e300f74faf6d1da23827d4c7e35ec890a
head ascii ...;.<...3..F%=.0.t....#.}L~5...
magic search PK -1 gzip 5586 7z -1 ctf= -1 flag{ -1

=== key 70 decoded len 8712
head hex e4dd7e37c7fff4ce183f87444a15fe9200cc78ca35ddeae08e4d8f72052f853a
head ascii ..~7.....?.DJ.....x.5....M.r./.:
magic search PK -1 gzip -1 7z -1 ctf= -1 flag{ -1

=== key 64 decoded len 8712
head hex fcbcf82fa679ecaf9e27e6c25274788a614a60abb3c58b66962c096a64a99d5b
head ascii .../.y...'..Rtx.aJ`....f.,.jd..[
magic search PK -1 gzip -1 7z -1 ctf= -1 flag{ -1

=== key 86 decoded len 8712
head hex c45f761745f7d44c101f054c6a97f6b282c458483dfd68e8aecf87528727a5b8
head ascii ._v.E..L...Lj.....XH=.h....R.'..
magic search PK 7594 gzip -1 7z -1 ctf= -1 flag{ -1

=== key 68 decoded len 8712
head hex ecfdfc3fe77dfcee9a37a7c642357c9a204e70eab7d5ca62866d0d7a25ad8d1a
head ascii ...?.}...7..B5|. Np....b.m.z%...
magic search PK -1 gzip -1 7z -1 ctf= -1 flag{ -1

=== key 71

In [41]:
# Extract the gzip stream found inside the 6-bit decoded blob for key 69
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
key = 69
x = bytes(b ^ key for b in data)

def symbol_from_byte(b: int) -> int:
    return ((b >> 4) & 0x7) * 8 + (b & 0x7)

def decode_6bit_stream(encoded: bytes) -> bytes:
    symbols = [symbol_from_byte(b) for b in encoded]
    out = bytearray()
    for i in range(0, len(symbols) - (len(symbols) % 4), 4):
        v = (symbols[i] << 18) | (symbols[i+1] << 12) | (symbols[i+2] << 6) | symbols[i+3]
        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
    return bytes(out)

decoded = decode_6bit_stream(x)
idx = decoded.find(b'\x1f\x8b')
print('gzip idx', idx)
if idx != -1:
    import gzip
    try:
        payload = gzip.decompress(decoded[idx:])
        print('decompressed len', len(payload))
        print('decompressed head:', payload[:256])
        with open('decoded_gzip_payload.bin', 'wb') as f:
            f.write(payload)
        print('wrote decoded_gzip_payload.bin')
    except Exception as e:
        print('gzip decompress error', repr(e))


gzip idx 5586
gzip decompress error BadGzipFile('Unknown compression method')


In [42]:
# Inspect the claimed gzip-like area in the decoded payload
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
key = 69
x = bytes(b ^ key for b in data)

def symbol_from_byte(b: int) -> int:
    return ((b >> 4) & 0x7) * 8 + (b & 0x7)

def decode_6bit_stream(encoded: bytes) -> bytes:
    symbols = [symbol_from_byte(b) for b in encoded]
    out = bytearray()
    for i in range(0, len(symbols) - (len(symbols) % 4), 4):
        v = (symbols[i] << 18) | (symbols[i+1] << 12) | (symbols[i+2] << 6) | symbols[i+3]
        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
    return bytes(out)

decoded = decode_6bit_stream(x)
idx = decoded.find(b'\x1f\x8b')
print('gzip-like idx', idx)
if idx != -1:
    snippet = decoded[idx:idx+32]
    print('hex:', snippet.hex())
    print('ascii:', ''.join(chr(b) if 32<=b<127 else '.' for b in snippet))
    print('byte values:', list(snippet))


gzip-like idx 5586
hex: 1f8be6cde02542e66f7adced7fd1f73153541a9d2f14a18dec08fe1c2c6ccaaf
ascii: .....%B.oz.....1ST../.......,l..
byte values: [31, 139, 230, 205, 224, 37, 66, 230, 111, 122, 220, 237, 127, 209, 247, 49, 83, 84, 26, 157, 47, 20, 161, 141, 236, 8, 254, 28, 44, 108, 202, 175]


In [43]:
# Search decoded 6-bit streams for common embedded file signatures and flag markers
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
keys = [69, 70, 64]
markers = [b'PK\x03\x04', b'7z\xbc\xaf\x27\x1c', b'\x1f\x8b', b'BZh', b'Rar!', b'MZ', b'%PDF', b'ctf=', b'flag{', b'http']

def symbol_from_byte(b: int) -> int:
    return ((b >> 4) & 0x7) * 8 + (b & 0x7)

def decode_6bit_stream(encoded: bytes) -> bytes:
    symbols = [symbol_from_byte(b) for b in encoded]
    out = bytearray()
    for i in range(0, len(symbols) - (len(symbols) % 4), 4):
        v = (symbols[i] << 18) | (symbols[i+1] << 12) | (symbols[i+2] << 6) | symbols[i+3]
        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
    return bytes(out)

for key in keys:
    x = bytes(b ^ key for b in data)
    decoded = decode_6bit_stream(x)
    print('\n=== key', key)
    for m in markers:
        idx = decoded.find(m)
        if idx != -1:
            print(' found', m, 'at', idx)
    # print some readable segments
    readable = b''.join(bytes([b]) if 32<=b<127 else b'.' for b in decoded[:200])
    print('head', readable[:200])



=== key 69
 found b'\x1f\x8b' at 5586
head b'...;.<...3..F%=.0.t....#.}L~5...z....MR..~.M.._.....J.-z..5..*-,....Q........9.....}.s .........<.....qU......l..8.9.....S2..X:.dp..CCy.~..h..d`K.....nAe:.-.+.7..8...Q.}...Y..H1..e&.%|..M.;b....9.XO..'

=== key 70
 found b'MZ' at 5453
head b'..~7.....?.DJ.....x.5....M.r./.:..3+.}.....}?.o$....z.!J.".W... .......S........b....C...l..b...0.a..7}e2..p........q..B.c....6..|.|Os..Nz.X..T.G.4..ebq.6.....;.l4.+.ajq*8.iT.x....*9.p..A..n.K....h...'

=== key 64
head b'.../.y...\'..Rtx.aJ`....f.,.jd..[?.R......;...........k9+.:d..{h8.k...........|.....8."e........R(.....e.......)..}.h......w.....!d..W.<./..9O.5%_.....z. ..h.zY#..,.....iK......t.. 2X`h..Y.~v....|.....'


In [44]:
# Extract and inspect the embedded PE file found in the key-70 decoded stream
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
key = 70
x = bytes(b ^ key for b in data)

def symbol_from_byte(b: int) -> int:
    return ((b >> 4) & 0x7) * 8 + (b & 0x7)

def decode_6bit_stream(encoded: bytes) -> bytes:
    symbols = [symbol_from_byte(b) for b in encoded]
    out = bytearray()
    for i in range(0, len(symbols) - (len(symbols) % 4), 4):
        v = (symbols[i] << 18) | (symbols[i+1] << 12) | (symbols[i+2] << 6) | symbols[i+3]
        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
    return bytes(out)

decoded = decode_6bit_stream(x)
print('decoded len', len(decoded))
for pos in range(len(decoded)):
    if decoded[pos:pos+2] == b'MZ':
        if pos + 0x40 <= len(decoded):
            e_lfanew = int.from_bytes(decoded[pos+0x3c:pos+0x40], 'little')
            pe_offset = pos + e_lfanew
            print('MZ at', pos, 'e_lfanew', e_lfanew, 'PE header', decoded[pe_offset:pe_offset+4])
            if decoded[pe_offset:pe_offset+4] == b'PE\x00\x00':
                print('  valid PE candidate at', pos)
                with open('decoded_pe_key70.bin', 'wb') as f:
                    f.write(decoded[pos:])
                print('  wrote decoded_pe_key70.bin')
                break


decoded len 8712
MZ at 5453 e_lfanew 930602134 PE header b''


In [45]:
# Test alternate 6-bit decoding order for the top candidate keys
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
keys = [69, 70]

for key in keys:
    x = bytes(b ^ key for b in data)
    print('\n=== key', key)
    for order in ['hi_lo', 'lo_hi']:
        if order == 'hi_lo':
            def symbol(b):
                return ((b >> 4) & 0x7) * 8 + (b & 0x7)
        else:
            def symbol(b):
                return (b & 0x7) * 8 + ((b >> 4) & 0x7)
        symbols = [symbol(b) for b in x]
        out = bytearray()
        for i in range(0, len(symbols) - (len(symbols) % 4), 4):
            v = (symbols[i] << 18) | (symbols[i+1] << 12) | (symbols[i+2] << 6) | symbols[i+3]
            out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
        decoded = bytes(out)
        sigs = {b'PK\x03\x04': decoded.find(b'PK\x03\x04'), b'\x1f\x8b': decoded.find(b'\x1f\x8b'), b'MZ': decoded.find(b'MZ'), b'7z\xbc\xaf\x27\x1c': decoded.find(b'7z\xbc\xaf\x27\x1c')}
        print(order, 'len', len(decoded), 'head hex', decoded[:32].hex(), 'sigs', sigs)



=== key 69
hi_lo len 8712 head hex e8edbd3bf73cf8fedb33b78746253d9e300f74faf6d1da23827d4c7e35ec890a sigs {b'PK\x03\x04': -1, b'\x1f\x8b': 5586, b'MZ': -1, b"7z\xbc\xaf'\x1c": -1}
lo_hi len 8712 head hex 5f1dafc7f8e7df97db85fcf82948aff1c039af97769ab15c13cba1edcea55023 sigs {b'PK\x03\x04': -1, b'\x1f\x8b': -1, b'MZ': 3019, b"7z\xbc\xaf'\x1c": -1}

=== key 70
hi_lo len 8712 head hex e4dd7e37c7fff4ce183f87444a15fe9200cc78ca35ddeae08e4d8f72052f853a sigs {b'PK\x03\x04': -1, b'\x1f\x8b': -1, b'MZ': 5453, b"7z\xbc\xaf'\x1c": -1}
lo_hi len 8712 head hex 3e9bb7a67effbe11c3e47ae048ceb7904621ce116efb3744724db98c48bd31a5 sigs {b'PK\x03\x04': -1, b'\x1f\x8b': -1, b'MZ': -1, b"7z\xbc\xaf'\x1c": -1}


In [46]:
# Inspect the MZ candidates from alternate 6-bit mapping
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()

for key, mapping in [(69, 'lo_hi'), (70, 'hi_lo')]:
    x = bytes(b ^ key for b in data)
    if mapping == 'hi_lo':
        def symbol(b):
            return ((b >> 4) & 0x7) * 8 + (b & 0x7)
    else:
        def symbol(b):
            return (b & 0x7) * 8 + ((b >> 4) & 0x7)
    symbols = [symbol(b) for b in x]
    out = bytearray()
    for i in range(0, len(symbols) - (len(symbols) % 4), 4):
        v = (symbols[i] << 18) | (symbols[i+1] << 12) | (symbols[i+2] << 6) | symbols[i+3]
        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
    decoded = bytes(out)
    idx = decoded.find(b'MZ')
    print('\n=== key', key, mapping, 'MZ idx', idx)
    print('head', decoded[idx:idx+64])
    if idx != -1 and idx+0x40 <= len(decoded):
        e_lfanew = int.from_bytes(decoded[idx+0x3c:idx+0x40], 'little')
        print('e_lfanew', e_lfanew, 'PE', decoded[idx+e_lfanew:idx+e_lfanew+4])



=== key 69 lo_hi MZ idx 3019
head b'MZ\xafOJK7-R\x9e\xad;\xfb\xb6\xd4\xfc\xb4\xdb(\xab\xe9\x11\x996\xedl\xbbI\x01.\xf6\x8e\x87\xe8@\x1e\x1a\xe1\x10_z\x8a\xd7-\xdaX"\xe0\x05\x93-\xbf\x0f\x9b\xe9\x00\x86\x9d\xe8\xcbG\xa3\xb8\xd8'
e_lfanew 3635979079 PE b''

=== key 70 hi_lo MZ idx 5453
head b'MZN\xee\x87\xe0\xeew\xe1A\x19\xcb\xa1\x17!\xd7E_\x9c\x84\x83\xd0\xc3\xb9\xc8<|\x0f\x15\x90\x94\xa0\x9e+\xcf\x87\xa1\xe9\x81&j\xb9\x00\xb4n\xade\x8b\xb2\xa0\x9f\x1e\xa6JW<O\xea\xa7\xcb\x96\xdcw7'
e_lfanew 930602134 PE b''


In [47]:
# Test 6-bit decoding alignment shifts for embedded signatures
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
keys = [69, 70]
markers = [b'PK\x03\x04', b'\x1f\x8b', b'MZ', b'ctf=', b'flag{']

for key in keys:
    x = bytes(b ^ key for b in data)
    print('\n=== key', key)
    for order in ['hi_lo', 'lo_hi']:
        if order == 'hi_lo':
            symbol = lambda b: ((b >> 4) & 0x7) * 8 + (b & 0x7)
        else:
            symbol = lambda b: (b & 0x7) * 8 + ((b >> 4) & 0x7)
        symbols = [symbol(b) for b in x]
        for shift in range(4):
            out = bytearray()
            for i in range(shift, len(symbols) - ((len(symbols)-shift) % 4), 4):
                v = (symbols[i] << 18) | (symbols[i+1] << 12) | (symbols[i+2] << 6) | symbols[i+3]
                out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
            decoded = bytes(out)
            hits = {m: decoded.find(m) for m in markers}
            if any(idx != -1 for idx in hits.values()):
                print(order, 'shift', shift, 'hits', hits)



=== key 69
hi_lo shift 0 hits {b'PK\x03\x04': -1, b'\x1f\x8b': 5586, b'MZ': -1, b'ctf=': -1, b'flag{': -1}
lo_hi shift 0 hits {b'PK\x03\x04': -1, b'\x1f\x8b': -1, b'MZ': 3019, b'ctf=': -1, b'flag{': -1}

=== key 70
hi_lo shift 0 hits {b'PK\x03\x04': -1, b'\x1f\x8b': -1, b'MZ': 5453, b'ctf=': -1, b'flag{': -1}
lo_hi shift 3 hits {b'PK\x03\x04': -1, b'\x1f\x8b': -1, b'MZ': 7774, b'ctf=': -1, b'flag{': -1}


In [48]:
# Decode using the actual 64-symbol alphabet order from the XOR result
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
keys = [69, 70]
markers = [b'PK\x03\x04', b'\x1f\x8b', b'MZ', b'ctf=', b'flag{']

for key in keys:
    x = bytes(b ^ key for b in data)
    alphabet = sorted(set(x))
    if len(alphabet) != 64:
        print('key', key, 'alphabet size', len(alphabet))
        continue
    mapping = {b: i for i, b in enumerate(alphabet)}
    symbols = [mapping[b] for b in x]
    out = bytearray()
    for i in range(0, len(symbols) - (len(symbols) % 4), 4):
        v = (symbols[i] << 18) | (symbols[i+1] << 12) | (symbols[i+2] << 6) | symbols[i+3]
        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
    decoded = bytes(out)
    print('\n=== key', key)
    print('alphabet', ''.join(chr(c) if 32 <= c < 127 else '.' for c in alphabet))
    print('len', len(decoded), 'head hex', decoded[:32].hex())
    print('marker positions', {m: decoded.find(m) for m in markers})
    if decoded.startswith(b'PK\x03\x04'):
        print('  starts with ZIP')
    if decoded.startswith(b'%PDF'):
        print('  starts with PDF')
    if decoded.startswith(b'7z\xbc\xaf\x27\x1c'):
        print('  starts with 7z')



=== key 69
alphabet ................ !"#$%&'01234567@ABCDEFGPQRSTUVW`abcdefgpqrstuvw
len 8712 head hex e8edbd3bf73cf8fedb33b78746253d9e300f74faf6d1da23827d4c7e35ec890a
marker positions {b'PK\x03\x04': -1, b'\x1f\x8b': 5586, b'MZ': -1, b'ctf=': -1, b'flag{': -1}

=== key 70
alphabet ................ !"#$%&'01234567@ABCDEFGPQRSTUVW`abcdefgpqrstuvw
len 8712 head hex e4dd7e37c7fff4ce183f87444a15fe9200cc78ca35ddeae08e4d8f72052f853a
marker positions {b'PK\x03\x04': -1, b'\x1f\x8b': -1, b'MZ': 5453, b'ctf=': -1, b'flag{': -1}


In [49]:
# Systematic search across top XOR keys, 6-bit mappings, and symbol alignment shifts
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
keys = [64, 66, 68, 69, 70, 71, 82, 86, 87, 117]

for key in keys:
    x = bytes(b ^ key for b in data)
    print('\n=== key', key)
    for order in ['hi_lo', 'lo_hi']:
        if order == 'hi_lo':
            symbol = lambda b: ((b >> 4) & 0x7) * 8 + (b & 0x7)
        else:
            symbol = lambda b: (b & 0x7) * 8 + ((b >> 4) & 0x7)
        symbols = [symbol(b) for b in x]
        for shift in range(4):
            out = bytearray()
            for i in range(shift, len(symbols) - ((len(symbols)-shift) % 4), 4):
                v = (symbols[i] << 18) | (symbols[i+1] << 12) | (symbols[i+2] << 6) | symbols[i+3]
                out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
            decoded = bytes(out)
            hits = {
                'PK': decoded.find(b'PK\x03\x04'),
                'gzip': decoded.find(b'\x1f\x8b'),
                'MZ': decoded.find(b'MZ'),
                '7z': decoded.find(b'7z\xbc\xaf\x27\x1c'),
                'ctf=': decoded.find(b'ctf='),
                'flag{': decoded.find(b'flag{')
            }
            if any(idx != -1 for idx in hits.values()):
                print(order, 'shift', shift, 'hits', hits)
                for name, idx in hits.items():
                    if idx != -1 and idx < 200:
                        head = decoded[idx:idx+64]
                        print('  ', name, 'head', head[:32], head[32:64])
                mz = hits['MZ']
                if mz != -1 and mz + 0x40 <= len(decoded):
                    e_lfanew = int.from_bytes(decoded[mz+0x3c:mz+0x40], 'little')
                    pe = decoded[mz+e_lfanew:mz+e_lfanew+4] if mz+e_lfanew+4 <= len(decoded) else None
                    print('   e_lfanew', e_lfanew, 'PE', pe)
                gz = hits['gzip']
                if gz != -1:
                    print('   gzip bytes', decoded[gz:gz+10].hex())



=== key 64

=== key 66
hi_lo shift 0 hits {'PK': -1, 'gzip': -1, 'MZ': 1463, '7z': -1, 'ctf=': -1, 'flag{': -1}
   e_lfanew 1188856363 PE None
lo_hi shift 0 hits {'PK': -1, 'gzip': -1, 'MZ': 65, '7z': -1, 'ctf=': -1, 'flag{': -1}
   MZ head b'MZ]\xf2\xd6\xf3z\xd2\xba\x9f>\xfew\xd6R\xf4;=\xd7\xf0&|y\xdb\x85P{t\xfd~\x82\x04' b'\x93\xeca\xf0\x9em \xb6\x95[\xa6\xba\xbc\x1d\xe0\xde\xffrC.\xf3s0]&n\xb8\x9c;&]\\'
   e_lfanew 1549608507 PE None
lo_hi shift 1 hits {'PK': -1, 'gzip': 6476, 'MZ': 749, '7z': -1, 'ctf=': -1, 'flag{': -1}
   e_lfanew 3124133133 PE None
   gzip bytes 1f8b26564b2db5213d45

=== key 68
hi_lo shift 1 hits {'PK': -1, 'gzip': 630, 'MZ': -1, '7z': -1, 'ctf=': -1, 'flag{': -1}
   gzip bytes 1f8b442f2d3bdb9056c6
hi_lo shift 2 hits {'PK': -1, 'gzip': 7732, 'MZ': -1, '7z': -1, 'ctf=': -1, 'flag{': -1}
   gzip bytes 1f8b4494d05a3e78bfca
lo_hi shift 2 hits {'PK': -1, 'gzip': 7130, 'MZ': -1, '7z': -1, 'ctf=': -1, 'flag{': -1}
   gzip bytes 1f8b78a4ecfba3b0303a

=== key 69
hi_lo s

In [52]:
# Broad search for the correct 6-bit packing/bit order and known file markers
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
keys = [64, 66, 68, 69, 70, 71, 82, 86, 87, 117]

mapping_funcs = {
    'hi_lo': lambda b: ((b >> 4) & 0x7) * 8 + (b & 0x7),
    'lo_hi': lambda b: (b & 0x7) * 8 + ((b >> 4) & 0x7),
}

rev6 = {i: int('{:06b}'.format(i)[::-1], 2) for i in range(64)}

pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}

markers = [b'PK\x03\x04', b'PK\x05\x06', b'PK\x07\x08', b'\x1f\x8b\x08', b'MZ', b'7z\xbc\xaf\x27\x1c', b'ctf=', b'flag{']

for key in keys:
    x = bytes(b ^ key for b in data)
    for order_name, sym_func in mapping_funcs.items():
        symbols = [sym_func(b) for b in x]
        if max(symbols) >= 64 or min(symbols) < 0:
            continue
        for shift in range(4):
            for rev in [False, True]:
                syms = symbols[shift:]
                if rev:
                    syms = [rev6[s] for s in syms]
                for pack_name, pack_func in pack_orders.items():
                    out = bytearray()
                    for i in range(0, len(syms) - (len(syms) % 4), 4):
                        v = pack_func(syms[i:i+4])
                        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
                    if not out:
                        continue
                    hit = {}
                    for m in markers:
                        idx = out.find(m)
                        if idx != -1:
                            hit[m] = idx
                    if hit:
                        print('\nkey', key, 'order', order_name, 'shift', shift, 'rev', rev, 'pack', pack_name, 'hits', hit)
                        if out.startswith(b'PK\x03\x04') or out.startswith(b'\x1f\x8b\x08') or out.startswith(b'MZ'):
                            print('  starts with', out[:16])
                        if b'MZ' in hit:
                            idx = hit[b'MZ']
                            if idx + 0x40 <= len(out):
                                e_lfanew = int.from_bytes(out[idx+0x3c:idx+0x40], 'little')
                                print('  e_lfanew', e_lfanew, 'PE', out[idx+e_lfanew:idx+e_lfanew+4] if idx+e_lfanew+4 <= len(out) else None)
                        if b'\x1f\x8b\x08' in hit:
                            idx = hit[b'\x1f\x8b\x08']
                            print('  gzip head', out[idx:idx+16].hex())
                        if b'PK\x03\x04' in hit:
                            idx = hit[b'PK\x03\x04']
                            print('  zip head', out[idx:idx+16].hex())



key 64 order hi_lo shift 3 rev True pack le hits {b'MZ': 5510}
  e_lfanew 2464776450 PE None

key 64 order lo_hi shift 0 rev True pack be hits {b'MZ': 6088}
  e_lfanew 1676479277 PE None

key 64 order lo_hi shift 1 rev True pack le hits {b'MZ': 183}
  e_lfanew 3137108032 PE None

key 66 order hi_lo shift 0 rev False pack be hits {b'MZ': 1463}
  e_lfanew 1188856363 PE None

key 66 order lo_hi shift 0 rev False pack be hits {b'MZ': 65}
  e_lfanew 1549608507 PE None

key 66 order lo_hi shift 1 rev False pack be hits {b'MZ': 749}
  e_lfanew 3124133133 PE None

key 68 order hi_lo shift 0 rev True pack le hits {b'MZ': 3356}
  e_lfanew 1696774419 PE None

key 68 order hi_lo shift 1 rev True pack be hits {b'MZ': 5860}
  e_lfanew 3101883144 PE None

key 68 order hi_lo shift 3 rev False pack le hits {b'MZ': 6869}
  e_lfanew 1795931748 PE None

key 68 order lo_hi shift 3 rev False pack le hits {b'MZ': 7358}
  e_lfanew 852537703 PE None

key 68 order lo_hi shift 3 rev True pack be hits {b'MZ': 30

In [70]:
# Search direct 6-bit decode candidates from raw bitplane_4_6_full.bin without pre-XOR
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
keys = [0]

mapping_funcs = {
    'hi_lo': lambda b: ((b >> 4) & 0x7) * 8 + (b & 0x7),
    'lo_hi': lambda b: (b & 0x7) * 8 + ((b >> 4) & 0x7),
}
rev6 = {i: int('{:06b}'.format(i)[::-1], 2) for i in range(64)}
pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}
markers = [b'PK\x03\x04', b'PK\x05\x06', b'PK\x07\x08', b'\x1f\x8b\x08', b'MZ', b'7z\xbc\xaf\x27\x1c', b'ctf=', b'flag{']

for key in keys:
    x = bytes(b ^ key for b in data)
    for order_name, sym_func in mapping_funcs.items():
        symbols = [sym_func(b) for b in x]
        for shift in range(4):
            for rev in [False, True]:
                syms = symbols[shift:]
                if rev:
                    syms = [rev6[s] for s in syms]
                for pack_name, pack_func in pack_orders.items():
                    out = bytearray()
                    for i in range(0, len(syms) - (len(syms) % 4), 4):
                        v = pack_func(syms[i:i+4])
                        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
                    for m in markers:
                        idx = out.find(m)
                        if idx != -1:
                            print('key', key, 'order', order_name, 'shift', shift, 'rev', rev, 'pack', pack_name, 'marker', m, 'offset', idx)


key 0 order hi_lo shift 2 rev True pack be marker b'MZ' offset 418
key 0 order lo_hi shift 1 rev False pack be marker b'MZ' offset 5189


In [71]:
# Inspect the two direct-decoded MZ candidates for valid PE headers
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()

mapping_funcs = {
    'hi_lo': lambda b: ((b >> 4) & 0x7) * 8 + (b & 0x7),
    'lo_hi': lambda b: (b & 0x7) * 8 + ((b >> 4) & 0x7),
}
rev6 = {i: int('{:06b}'.format(i)[::-1], 2) for i in range(64)}
pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}

candidates = [
    (0, 'hi_lo', 2, True, 'be', 418),
    (0, 'lo_hi', 1, False, 'be', 5189),
]

for key, order_name, shift, rev, pack_name, offset in candidates:
    x = data
    symbols = [mapping_funcs[order_name](b) for b in x]
    if rev:
        symbols = [rev6[s] for s in symbols[shift:]]
    else:
        symbols = symbols[shift:]
    out = bytearray()
    for i in range(0, len(symbols) - (len(symbols) % 4), 4):
        v = pack_orders[pack_name](symbols[i:i+4])
        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
    print('candidate', order_name, shift, rev, pack_name, 'len', len(out), 'offset', offset)
    header = out[offset:offset+16]
    print('header', header.hex())
    if offset + 0x40 <= len(out):
        e_lfanew = int.from_bytes(out[offset+0x3c:offset+0x40], 'little')
        print('e_lfanew', e_lfanew)
        if offset + e_lfanew + 4 <= len(out):
            print('PE bytes', out[offset+e_lfanew:offset+e_lfanew+4])
    print('window', out[offset-16:offset+64].hex())
    print('-'*60)


candidate hi_lo 2 True be len 8709 offset 418
header 4d5ab9ca930da1ee9b4a440bc39dd100
e_lfanew 1689285116
window 188d050acf860460b92b59a6323c9f6d4d5ab9ca930da1ee9b4a440bc39dd100e25a6aa4e9ec9bc08384298cbe911662d506c59e05762241acac19695fe20febee80321e8c6ff7e25c2652a9fc71b064
------------------------------------------------------------
candidate lo_hi 1 False be len 8712 offset 5189
header 4d5ab6966328667245426a2cb95dbeaf
e_lfanew 1503863301
window 5bc62930bdbf2ee25c14c9fcfbe2054b4d5ab6966328667245426a2cb95dbeaf14b2968325481c8d5c63c53669c2cb2f7174bffc0267fe7b07c1b0862c5da6a0e4055a109892d856ce1a48a80522a359
------------------------------------------------------------


In [76]:
# Search raw symbol grouping permutations for gzip/zip/7z/ctf/flag headers in direct decode outputs
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()

def make_perm(offset, reverse=False):
    if not reverse:
        return [(i + offset) % 8 for i in range(8)]
    else:
        return [(-i + offset) % 8 for i in range(8)]

mapping_funcs = {
    'hi_lo': lambda b: ((b >> 4) & 0x7) * 8 + (b & 0x7),
    'lo_hi': lambda b: (b & 0x7) * 8 + ((b >> 4) & 0x7),
}
rev6 = {i: int('{:06b}'.format(i)[::-1], 2) for i in range(64)}
pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}
markers = [b'\x1f\x8b\x08', b'PK\x03\x04', b'PK\x05\x06', b'PK\x07\x08', b'7z\xbc\xaf\x27\x1c', b'ctf=', b'flag{']

for mapping_name, sym_func in mapping_funcs.items():
    raw_symbols = [sym_func(b) for b in data]
    for offset in range(8):
        for rev_group in [False, True]:
            perm = make_perm(offset, reverse=rev_group)
            if False:
                pass
            for low_rev in [False, True]:
                values = []
                for b in data:
                    g = b >> 4
                    if g >= 8:
                        values = None
                        break
                    low = b & 7
                    lb = 7 - low if low_rev else low
                    values.append((perm[g] << 3) | lb)
                if values is None:
                    continue
                for shift in range(4):
                    for bitrev in [False, True]:
                        syms = values[shift:]
                        if bitrev:
                            syms = [rev6[s] for s in syms]
                        for pack_name, pack_func in pack_orders.items():
                            out = bytearray()
                            for i in range(0, len(syms) - (len(syms) % 4), 4):
                                v = pack_func(syms[i:i+4])
                                out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
                            for m in markers:
                                idx = out.find(m)
                                if idx != -1:
                                    print('mapping', mapping_name, 'perm_offset', offset, 'rev_group', rev_group, 'low_rev', low_rev, 'shift', shift, 'bitrev', bitrev, 'pack', pack_name, 'marker', m, 'offset', idx)
print('done search')


done search


In [78]:
# Decode using the discovered group permutation candidate and inspect the gzip payload
import gzip
import itertools

fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()

perm = (0, 6, 1, 2, 5, 3, 4, 7)
shift = 1
bitrev = False
pack_name = 'le'
markers = [b'\x1f\x8b\x08']

values = []
for b in data:
    g = b >> 4
    low = b & 7
    values.append((perm[g] << 3) | low)
syms = values[shift:]
if bitrev:
    syms = [int('{:06b}'.format(v)[::-1], 2) for v in syms]
out = bytearray()
for i in range(0, len(syms) - (len(syms) % 4), 4):
    v = (syms[i+3] << 18) | (syms[i+2] << 12) | (syms[i+1] << 6) | syms[i]
    out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
print('len', len(out))
idx = out.find(b'\x1f\x8b\x08')
print('gzip offset', idx)
print('header', out[idx:idx+16].hex())
print('window', out[idx-16:idx+32].hex())
try:
    decompressed = gzip.decompress(out[idx:])
    print('decompressed len', len(decompressed))
    print(decompressed[:256])
    open('decoded_gzip_payload.bin', 'wb').write(out[idx:])
    print('wrote decoded_gzip_payload.bin')
except Exception as e:
    print('decompress failed', type(e).__name__, e)


len 8712
gzip offset 7040
header 1f8b08a38fb596d21d5721ff2800e0e1
window b9e5ab0128847cb02203955e7e9e003c1f8b08a38fb596d21d5721ff2800e0e1cf3773b710192e8d21db36e1d4732dce
decompress failed error Error -3 while decompressing data: invalid stored block lengths


In [87]:
# Decode using the discovered bit-position permutation candidate and inspect the gzip payload
import gzip

fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
perm = (0, 4, 2, 5, 1, 6)
shift = 1
bitrev = True
pack_name = 'le'

values = []
for b in data:
    v = 0
    for i, bit_pos in enumerate(perm):
        v |= ((b >> bit_pos) & 1) << i
    values.append(v)
syms = values[shift:]
if bitrev:
    syms = [int('{:06b}'.format(v)[::-1], 2) for v in syms]
out = bytearray()
for i in range(0, len(syms) - (len(syms) % 4), 4):
    v = (syms[i+3] << 18) | (syms[i+2] << 12) | (syms[i+1] << 6) | syms[i]
    out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
idx = out.find(b'\x1f\x8b\x08')
print('gzip offset', idx)
print('header', out[idx:idx+16].hex())
print('window', out[idx-16:idx+32].hex())
try:
    decompressed = gzip.decompress(out[idx:])
    print('decompressed len', len(decompressed))
    print(decompressed[:256])
except Exception as e:
    print('decompress failed', type(e).__name__, e)


gzip offset 2107
header 1f8b08f40811730e75047aa83d2c52bd
window b404d1933085434393bec320567b41bd1f8b08f40811730e75047aa83d2c52bd0631dc57de107ecb97406056d9028bfd
decompress failed EOFError Compressed file ended before the end-of-stream marker was reached


In [88]:
# Search all bit-position permutations and try decompressing any found gzip stream
import itertools
import gzip

fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()

slots = [0, 1, 2, 4, 5, 6]
pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}

for perm in itertools.permutations(slots):
    values = []
    for b in data:
        v = 0
        for i, bit_pos in enumerate(perm):
            v |= ((b >> bit_pos) & 1) << i
        values.append(v)
    for shift in range(4):
        for bitrev in [False, True]:
            if bitrev:
                syms = [int('{:06b}'.format(v)[::-1], 2) for v in values[shift:]]
            else:
                syms = values[shift:]
            for pack_name, pack_func in pack_orders.items():
                out = bytearray()
                for i in range(0, len(syms) - (len(syms) % 4), 4):
                    v = pack_func(syms[i:i+4])
                    out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
                idx = out.find(b'\x1f\x8b\x08')
                if idx != -1:
                    try:
                        decompressed = gzip.decompress(out[idx:])
                        print('SUCCESS', perm, 'shift', shift, 'bitrev', bitrev, 'pack', pack_name, 'idx', idx, 'len', len(decompressed))
                        print(decompressed[:256])
                        raise SystemExit
                    except Exception as e:
                        print('FAIL', perm, 'shift', shift, 'bitrev', bitrev, 'pack', pack_name, 'idx', idx, type(e).__name__, e)
print('done')


FAIL (0, 4, 2, 5, 1, 6) shift 1 bitrev True pack le idx 2107 EOFError Compressed file ended before the end-of-stream marker was reached
FAIL (4, 0, 2, 5, 1, 6) shift 1 bitrev True pack le idx 2107 error Error -3 while decompressing data: invalid code lengths set
FAIL (6, 1, 5, 2, 0, 4) shift 1 bitrev False pack le idx 2107 error Error -3 while decompressing data: invalid code lengths set
FAIL (6, 1, 5, 2, 4, 0) shift 1 bitrev False pack le idx 2107 EOFError Compressed file ended before the end-of-stream marker was reached
done


In [91]:
# Search bit-position permutations for valid gzip headers with correct flag bits
import itertools
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
slots = [0, 1, 2, 4, 5, 6]
pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}

for perm in itertools.permutations(slots):
    values = []
    for b in data:
        v = 0
        for i, bit_pos in enumerate(perm):
            v |= ((b >> bit_pos) & 1) << i
        values.append(v)
    for shift in range(4):
        for bitrev in [False, True]:
            syms = values[shift:]
            if bitrev:
                syms = [int('{:06b}'.format(v)[::-1], 2) for v in syms]
            for pack_name, pack_func in pack_orders.items():
                out = bytearray()
                for i in range(0, len(syms) - (len(syms) % 4), 4):
                    v = pack_func(syms[i:i+4])
                    out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
                idx = 0
                while True:
                    idx = out.find(b'\x1f\x8b\x08', idx)
                    if idx == -1:
                        break
                    flags = out[idx+3]
                    if flags & 0xE0 == 0:
                        print('VALID gzip', perm, 'shift', shift, 'bitrev', bitrev, 'pack', pack_name, 'offset', idx, 'flags', flags)
                    idx += 1
print('search done')


search done


In [92]:
# Search bit-position permutations for valid PE headers in direct decode outputs
import itertools

fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
slots = [0, 1, 2, 4, 5, 6]
pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}

for perm in itertools.permutations(slots):
    values = []
    for b in data:
        v = 0
        for i, bit_pos in enumerate(perm):
            v |= ((b >> bit_pos) & 1) << i
        values.append(v)
    for shift in range(4):
        for bitrev in [False, True]:
            syms = values[shift:]
            if bitrev:
                syms = [int('{:06b}'.format(v)[::-1], 2) for v in syms]
            for pack_name, pack_func in pack_orders.items():
                out = bytearray()
                for i in range(0, len(syms) - (len(syms) % 4), 4):
                    v = pack_func(syms[i:i+4])
                    out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
                idx = 0
                while True:
                    idx = out.find(b'MZ', idx)
                    if idx == -1:
                        break
                    if idx + 0x40 <= len(out):
                        e_lfanew = int.from_bytes(out[idx+0x3c:idx+0x40], 'little')
                        if idx + e_lfanew + 4 <= len(out) and out[idx+e_lfanew:idx+e_lfanew+4] == b'PE\x00\x00':
                            print('VALID PE', perm, 'shift', shift, 'bitrev', bitrev, 'pack', pack_name, 'MZ offset', idx, 'e_lfanew', e_lfanew)
                    idx += 1
print('PE search done')


PE search done


In [95]:
# Search bit-position permutations for zlib headers and test decompression
import itertools
import zlib

fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
slots = [0, 1, 2, 4, 5, 6]
pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}

for perm in itertools.permutations(slots):
    values = []
    for b in data:
        v = 0
        for i, bit_pos in enumerate(perm):
            v |= ((b >> bit_pos) & 1) << i
        values.append(v)
    for shift in range(4):
        for bitrev in [False, True]:
            syms = values[shift:]
            if bitrev:
                syms = [int('{:06b}'.format(v)[::-1], 2) for v in syms]
            for pack_name, pack_func in pack_orders.items():
                out = bytearray()
                for i in range(0, len(syms) - (len(syms) % 4), 4):
                    v = pack_func(syms[i:i+4])
                    out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
                for header in [b'\x78\x01', b'\x78\x9c', b'\x78\xda']:
                    idx = out.find(header)
                    if idx != -1:
                        try:
                            decompressed = zlib.decompress(out[idx:])
                            print('SUCCESS', perm, 'shift', shift, 'bitrev', bitrev, 'pack', pack_name, 'header', header, 'idx', idx, 'len', len(decompressed))
                            print(decompressed[:256])
                            raise SystemExit
                        except Exception:
                            pass
print('search done')


search done


In [96]:
# Search bit-position permutations plus 6-bit XOR for gzip or zlib headers
import itertools
import zlib

fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
slots = [0, 1, 2, 4, 5, 6]
pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}

for perm in itertools.permutations(slots):
    values = []
    for b in data:
        v = 0
        for i, bit_pos in enumerate(perm):
            v |= ((b >> bit_pos) & 1) << i
        values.append(v)
    for xor6 in range(64):
        xor_values = [(v ^ xor6) & 0x3F for v in values]
        for shift in range(4):
            for bitrev in [False, True]:
                syms = xor_values[shift:]
                if bitrev:
                    syms = [int('{:06b}'.format(v)[::-1], 2) for v in syms]
                for pack_name, pack_func in pack_orders.items():
                    out = bytearray()
                    for i in range(0, len(syms) - (len(syms) % 4), 4):
                        v = pack_func(syms[i:i+4])
                        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
                    # valid gzip header by flags bits
                    idx = out.find(b'\x1f\x8b\x08')
                    while idx != -1:
                        flags = out[idx+3]
                        if flags & 0xE0 == 0:
                            try:
                                decompressed = zlib.decompress(out[idx+2:])
                            except Exception:
                                decompressed = None
                            if decompressed is not None:
                                print('SUCCESS gzip', perm, xor6, shift, bitrev, pack_name, idx, 'len', len(decompressed))
                                print(decompressed[:256])
                                raise SystemExit
                        idx = out.find(b'\x1f\x8b\x08', idx+1)
                    # zlib headers
                    for header in [b'\x78\x01', b'\x78\x9c', b'\x78\xda']:
                        idx = out.find(header)
                        if idx != -1:
                            try:
                                decompressed = zlib.decompress(out[idx:])
                                print('SUCCESS zlib', perm, xor6, shift, bitrev, pack_name, header, idx, 'len', len(decompressed))
                                print(decompressed[:256])
                                raise SystemExit
                            except Exception:
                                pass
print('done search')


KeyboardInterrupt: 

In [89]:
# Test a handful of gzip-found candidates with all single-byte XORs on the packed output
import gzip

fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()

candidates = [
    ((0, 4, 2, 5, 1, 6), 1, True, 'le'),
    ((4, 0, 2, 5, 1, 6), 1, True, 'le'),
    ((6, 1, 5, 2, 0, 4), 1, False, 'le'),
    ((6, 1, 5, 2, 4, 0), 1, False, 'le'),
]
pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}

for perm, shift, bitrev, pack_name in candidates:
    values = []
    for b in data:
        v = 0
        for i, bit_pos in enumerate(perm):
            v |= ((b >> bit_pos) & 1) << i
        values.append(v)
    syms = values[shift:]
    if bitrev:
        syms = [int('{:06b}'.format(v)[::-1], 2) for v in syms]
    out = bytearray()
    for i in range(0, len(syms) - (len(syms) % 4), 4):
        v = pack_orders[pack_name](syms[i:i+4])
        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
    print('candidate', perm, shift, bitrev, pack_name, 'len', len(out))
    found = False
    for xor_key in range(256):
        xo = bytes(b ^ xor_key for b in out)
        idx = xo.find(b'\x1f\x8b\x08')
        if idx == -1:
            continue
        try:
            decompressed = gzip.decompress(xo[idx:])
            print('  SUCCESS xor', xor_key, 'idx', idx, 'len', len(decompressed))
            print(decompressed[:256])
            found = True
            break
        except Exception as e:
            pass
    if not found:
        print('  no valid gzip under xor 0..255')


candidate (0, 4, 2, 5, 1, 6) 1 True le len 8712
  no valid gzip under xor 0..255
candidate (4, 0, 2, 5, 1, 6) 1 True le len 8712
  no valid gzip under xor 0..255
candidate (6, 1, 5, 2, 0, 4) 1 False le len 8712
  no valid gzip under xor 0..255
candidate (6, 1, 5, 2, 4, 0) 1 False le len 8712
  no valid gzip under xor 0..255


In [90]:
# Inspect ASCII runs and entropy of the candidate decoded output
import re

fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
perm = (0, 4, 2, 5, 1, 6)
shift = 1
bitrev = True
pack_name = 'le'

values = []
for b in data:
    v = 0
    for i, bit_pos in enumerate(perm):
        v |= ((b >> bit_pos) & 1) << i
    values.append(v)
syms = values[shift:]
if bitrev:
    syms = [int('{:06b}'.format(v)[::-1], 2) for v in syms]
out = bytearray()
for i in range(0, len(syms) - (len(syms) % 4), 4):
    v = (syms[i+3] << 18) | (syms[i+2] << 12) | (syms[i+1] << 6) | syms[i]
    out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])

def ascii_runs(blob, min_len=8):
    runs = []
    s = re.compile(rb'[ -~]{%d,}' % min_len)
    for m in s.finditer(blob):
        runs.append((m.start(), m.group().decode('ascii', errors='ignore')))
    return runs

print('output len', len(out))
print('first 64 bytes', out[:64].hex())
print('printable ratio', sum(32 <= b < 127 for b in out) / len(out))
runs = ascii_runs(out, 6)
print('ascii runs', runs[:20])


output len 8712
first 64 bytes cd49b3db4d56c5f793343dde09496a553a4a926693a0ae155f1102ab01ca83e62d8784ec8af124cb647b8741a40c2585d3309e6f2e206e96871a59ca6d0e2a13
printable ratio 0.3782139577594123
ascii runs [(496, 'Ezq].S'), (3677, '9SPps?'), (4029, 'C(/W"?O'), (4431, 'orbc)@k9'), (5747, '*(OS@_'), (6780, '4docF`'), (6840, 'H 3U!c'), (7148, 'qP`DRO'), (8690, 'O\\LT?)E')]


In [81]:
# Search direct 6-bit decoded output with a 6-bit XOR key applied to symbol values before packing
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()

markers = [b'\x1f\x8b\x08', b'PK\x03\x04', b'PK\x05\x06', b'PK\x07\x08', b'7z\xbc\xaf\x27\x1c', b'ctf=', b'flag{']

for mapping_name, sym_func in [('hi_lo', lambda b: ((b >> 4) & 0x7) * 8 + (b & 0x7)),
                               ('lo_hi', lambda b: (b & 0x7) * 8 + ((b >> 4) & 0x7))]:
    symbols = [sym_func(b) for b in data]
    for xor6 in range(64):
        xored = [(s ^ xor6) & 0x3F for s in symbols]
        for shift in range(4):
            for bitrev in [False, True]:
                syms = xored[shift:]
                if bitrev:
                    syms = [int('{:06b}'.format(v)[::-1], 2) for v in syms]
                for pack_name, pack_func in [('be', lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3])),
                                            ('le', lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]))]:
                    out = bytearray()
                    for i in range(0, len(syms) - (len(syms) % 4), 4):
                        v = pack_func(syms[i:i+4])
                        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
                    for m in markers:
                        if out.find(m) != -1:
                            print('mapping', mapping_name, 'xor6', xor6, 'shift', shift, 'bitrev', bitrev, 'pack', pack_name, 'marker', m)
    print('done mapping', mapping_name)


done mapping hi_lo
done mapping lo_hi


In [82]:
# Try interpreting the 6-bit symbol values as a Base64-style text stream
import base64

fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()

alphabet = b'ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz0123456789+/'

for mapping_name, sym_func in [('hi_lo', lambda b: ((b>>4)&7)*8 + (b&7)), ('lo_hi', lambda b: (b&7)*8 + ((b>>4)&7))]:
    values = [sym_func(b) for b in data]
    text = bytes(alphabet[v] for v in values)
    padded = text + b'=' * ((4 - len(text) % 4) % 4)
    print('mapping', mapping_name, 'first64', text[:64])
    try:
        decoded = base64.b64decode(padded, validate=True)
        print('base64 decode len', len(decoded))
        print(decoded[:256])
    except Exception as e:
        print('base64 decode failed:', type(e).__name__, e)
    print()

mapping hi_lo first64 b'frTYra5Zbqe+pe7i0HxYCGlq4qOTR4NGFCQp6GyJH1MfBVqNhRQ3DvobLRSZhwaC'
base64 decode failed: Error Invalid base64-encoded string: number of data characters (11617) cannot be 1 more than a multiple of 4

mapping lo_hi first64 b'7daDdTPLbVz3NzfUm4ODQwsVHVxaKHpwoQCNXwWJ4uh7IqVpMKC+Y9FbZKSLMGTQ'
base64 decode failed: Error Invalid base64-encoded string: number of data characters (11617) cannot be 1 more than a multiple of 4



In [84]:
# Confirm candidate_payload.bin exists and inspect its size
import os
for path in ['candidate_payload.bin', 'decoded_gzip_payload.bin']:
    print(path, os.path.exists(path), os.path.getsize(path) if os.path.exists(path) else None)


candidate_payload.bin False None
decoded_gzip_payload.bin False None


In [85]:
import os
print('cwd', os.getcwd())
print('files', os.listdir('.'))


cwd /home/diomonogatari/Projects/Arcus
files ['bitplane_4_1_alg_3.bin', 'test_align_14.bin', 'test_align_3.bin', 'blob4.bin', 'bitplane_8_1.bin', 'test_align_6.bin', 'bitplane_4_1_alg_17.bin', 'bitplane_4_1_fix.bin', 'ode_payload_analysis.ipynb', 'bitplane_4_1_alg_116.bin', 'detect.py', 'test_align_8.bin', 'bitplane_4_1_alg_108.bin', 'test_align_9.bin', 'bitplane_4_1_alg_106.bin', 'bitplane_4_6.bin', 'bitplane_4_1_full_xor1.bin', 'bitplane_4_1_alg_107.bin', 'arcus-artifacts-ode-triunfal-v1.zip', 'bitplane_4_1.bin', 'test_align_7.bin', 'bitplane_8_6.bin', 'blob8.bin', 'ctf_analysis.md', 'test_align_1.bin', 'test_align_13.bin', 'ode_analysis.ipynb', 'requirements.txt', 'bitplane_4_1_alg_20.bin', 'bitplane_4_1_alg_100.bin', 'bitplane_4_1_alg_16.bin', 'test_align_11.bin', 'bitplane_4_1_alg_109.bin', 'bitplane_4_1_full.bin', 'bitplane_4_1_alg_102.bin', 'bitplane_4_1_alg_19.bin', 'bitplane_4_1_alg_104.bin', 'bitplane_4_1_alg_103.bin', 'bitplane_4_1_alg_2.bin', 'final.py', 'bitplane_4_6_full.

In [73]:
# Search for known headers in direct 6-bit decoded outputs after single-byte XOR
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()

mapping_funcs = {
    'hi_lo': lambda b: ((b >> 4) & 0x7) * 8 + (b & 0x7),
    'lo_hi': lambda b: (b & 0x7) * 8 + ((b >> 4) & 0x7),
}
rev6 = {i: int('{:06b}'.format(i)[::-1], 2) for i in range(64)}
pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}

candidates = [
    ('hi_lo', 2, True, 'be'),
    ('lo_hi', 1, False, 'be'),
]
markers = [b'\x1f\x8b\x08', b'PK\x03\x04', b'PK\x05\x06', b'PK\x07\x08', b'7z\xbc\xaf\x27\x1c', b'MZ', b'ctf=', b'flag{']

for mapping_name, shift, rev, pack_name in candidates:
    sym_func = mapping_funcs[mapping_name]
    symbols = [sym_func(b) for b in data]
    if rev:
        symbols = [rev6[s] for s in symbols[shift:]]
    else:
        symbols = symbols[shift:]
    out = bytearray()
    for i in range(0, len(symbols) - (len(symbols) % 4), 4):
        v = pack_orders[pack_name](symbols[i:i+4])
        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
    print('candidate', mapping_name, shift, rev, pack_name, 'len', len(out))
    for xor_key in range(256):
        xo = bytes(b ^ xor_key for b in out)
        for m in markers:
            idx = xo.find(m)
            if idx != -1:
                print('  xor', xor_key, 'found', m, 'at', idx)
    print()

candidate hi_lo 2 True be len 8709
  xor 0 found b'MZ' at 418
  xor 9 found b'MZ' at 7028
  xor 14 found b'MZ' at 2167
  xor 27 found b'MZ' at 6470
  xor 30 found b'MZ' at 557
  xor 32 found b'MZ' at 2957
  xor 60 found b'MZ' at 8628
  xor 65 found b'MZ' at 3836
  xor 83 found b'MZ' at 5963
  xor 84 found b'MZ' at 4651
  xor 87 found b'MZ' at 8266
  xor 91 found b'MZ' at 743
  xor 96 found b'MZ' at 6642
  xor 101 found b'MZ' at 8029
  xor 109 found b'MZ' at 1708
  xor 110 found b'MZ' at 3553
  xor 117 found b'MZ' at 8019
  xor 128 found b'MZ' at 6119
  xor 137 found b'MZ' at 7009
  xor 138 found b'MZ' at 296
  xor 143 found b'MZ' at 189
  xor 147 found b'MZ' at 4609
  xor 161 found b'MZ' at 686
  xor 171 found b'MZ' at 482
  xor 173 found b'MZ' at 6422
  xor 175 found b'MZ' at 6565
  xor 191 found b'\x1f\x8b\x08' at 7054
  xor 204 found b'MZ' at 779
  xor 232 found b'MZ' at 8509
  xor 240 found b'MZ' at 3413

candidate lo_hi 1 False be len 8712
  xor 0 found b'MZ' at 5189
  xor 15 foun

In [75]:
# Evaluate all XOR keys for gzip headers in direct decode candidate outputs
import gzip
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()

mapping_funcs = {
    'hi_lo': lambda b: ((b >> 4) & 0x7) * 8 + (b & 0x7),
    'lo_hi': lambda b: (b & 0x7) * 8 + ((b >> 4) & 0x7),
}
rev6 = {i: int('{:06b}'.format(i)[::-1], 2) for i in range(64)}
pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}

candidates = [
    ('hi_lo', 2, True, 'be'),
    ('lo_hi', 1, False, 'be'),
]

def build_out(mapping_name, shift, rev, pack_name):
    sym_func = mapping_funcs[mapping_name]
    symbols = [sym_func(b) for b in data]
    if rev:
        symbols = [rev6[s] for s in symbols[shift:]]
    else:
        symbols = symbols[shift:]
    out = bytearray()
    for i in range(0, len(symbols) - (len(symbols) % 4), 4):
        v = pack_orders[pack_name](symbols[i:i+4])
        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
    return bytes(out)

for mapping_name, shift, rev, pack_name in candidates:
    out = build_out(mapping_name, shift, rev, pack_name)
    print('candidate', mapping_name, shift, rev, pack_name)
    for xor_key in range(256):
        xo = bytes(b ^ xor_key for b in out)
        idx = xo.find(b'\x1f\x8b\x08')
        if idx != -1:
            try:
                decompressed = gzip.decompress(xo[idx:])
                print('  SUCCESS key', xor_key, 'idx', idx, 'len', len(decompressed))
                print(decompressed[:128])
                break
            except Exception as e:
                print('  FAIL key', xor_key, 'idx', idx, type(e).__name__, e)
    print()

candidate hi_lo 2 True be
  FAIL key 191 idx 7054 error Error -3 while decompressing data: invalid block type

candidate lo_hi 1 False be



In [53]:
# Evaluate candidate 6-bit decode outputs for valid PE and gzip headers
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
keys = [64, 66, 68, 69, 70, 71, 82, 86, 87, 117]

mapping_funcs = {
    'hi_lo': lambda b: ((b >> 4) & 0x7) * 8 + (b & 0x7),
    'lo_hi': lambda b: (b & 0x7) * 8 + ((b >> 4) & 0x7),
}
rev6 = {i: int('{:06b}'.format(i)[::-1], 2) for i in range(64)}
pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}

for key in keys:
    x = bytes(b ^ key for b in data)
    for order_name, sym_func in mapping_funcs.items():
        symbols = [sym_func(b) for b in x]
        if max(symbols) >= 64 or min(symbols) < 0:
            continue
        for shift in range(4):
            for rev in [False, True]:
                syms = symbols[shift:]
                if rev:
                    syms = [rev6[s] for s in syms]
                for pack_name, pack_func in pack_orders.items():
                    out = bytearray()
                    for i in range(0, len(syms) - (len(syms) % 4), 4):
                        v = pack_func(syms[i:i+4])
                        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
                    if out.startswith(b'MZ'):
                        if len(out) >= 0x40:
                            e_lfanew = int.from_bytes(out[0x3c:0x40], 'little')
                            valid_pe = out[e_lfanew:e_lfanew+4] == b'PE\x00\x00' if e_lfanew < len(out) else False
                        else:
                            e_lfanew = None
                            valid_pe = False
                        print('MZ start found key', key, order_name, 'shift', shift, 'rev', rev, 'pack', pack_name, 'e_lfanew', e_lfanew, 'valid PE', valid_pe)
                    if b'\x1f\x8b\x08' in out[:16]:
                        print('gzip start found key', key, order_name, 'shift', shift, 'rev', rev, 'pack', pack_name, 'head', out[:16].hex())
                    if out.startswith(b'PK\x03\x04'):
                        print('zip start found key', key, order_name, 'shift', shift, 'rev', rev, 'pack', pack_name, 'head', out[:16].hex())
                    if b'ctf=' in out[:128] or b'flag{' in out[:128]:
                        print('flag-like start found key', key, order_name, 'shift', shift, 'rev', rev, 'pack', pack_name, 'prefix', out[:64])


In [56]:
# Collect marker hits and dump context for candidate 6-bit decode outputs (limited results)
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
keys = [64, 66, 68, 69, 70, 71, 82, 86, 87, 117]

mapping_funcs = {
    'hi_lo': lambda b: ((b >> 4) & 0x7) * 8 + (b & 0x7),
    'lo_hi': lambda b: (b & 0x7) * 8 + ((b >> 4) & 0x7),
}
rev6 = {i: int('{:06b}'.format(i)[::-1], 2) for i in range(64)}
pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}
markers = [b'PK\x03\x04', b'PK\x05\x06', b'PK\x07\x08', b'\x1f\x8b\x08', b'MZ', b'7z\xbc\xaf\x27\x1c', b'ctf=', b'flag{']

count = 0
for key in keys:
    x = bytes(b ^ key for b in data)
    for order_name, sym_func in mapping_funcs.items():
        symbols = [sym_func(b) for b in x]
        if max(symbols) >= 64 or min(symbols) < 0:
            continue
        for shift in range(4):
            for rev in [False, True]:
                syms = symbols[shift:]
                if rev:
                    syms = [rev6[s] for s in syms]
                for pack_name, pack_func in pack_orders.items():
                    out = bytearray()
                    for i in range(0, len(syms) - (len(syms) % 4), 4):
                        v = pack_func(syms[i:i+4])
                        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
                    for m in markers:
                        idx = out.find(m)
                        if idx != -1 and count < 20:
                            context = out[max(0, idx-16):idx+64]
                            print('key', key, 'order', order_name, 'shift', shift, 'rev', rev, 'pack', pack_name, 'marker', m, 'offset', idx, 'context', context.hex())
                            count += 1
                        elif idx != -1:
                            count += 1
                            if count == 20:
                                print('...more hits suppressed... total hits:', count)
                                break
                    if count >= 20:
                        break
                if count >= 20:
                    break
            if count >= 20:
                break
        if count >= 20:
            break
    if count >= 20:
        break
print('total hits seen (limited):', count)


key 64 order hi_lo shift 3 rev True pack le marker b'MZ' offset 5510 context 0d5815efb55fc74bf51f469d0922157f4d5a08c2c91b23563ccca65b8f2fb652be2f36d45077e07da86bc3209b59aa579af68facb692232fa7b9f29d25e1083c801dd10c1f8438467a3535030281e992
key 64 order lo_hi shift 0 rev True pack be marker b'MZ' offset 6088 context 8387d29877e6c98291666034233c14e34d5a40a43a62070ff902a1d6ba77b1280281a41641e2e49dba765e6e7d08a84e5a1f8416aa3938c957fe828f66ae365be9b388e196ea3c092c7c8bd92d0bed63
key 64 order lo_hi shift 1 rev True pack le marker b'MZ' offset 183 context 88f7bad109238031934c84802ec8830d4d5ae0d7bd70aa53aa73932be22258872228a3a92ec983df5f1e2a9100df1812da95fc9cc13f6d3d5cbc4eb2a33a8488a8e35423dc9d0cc32e53f2844078fcba
key 66 order hi_lo shift 0 rev False pack be marker b'MZ' offset 1463 context d8e762a6e6719a6ef6bf1faeebc35e384d5a9aa48d84870c850787a0686d7c37826af8cb9849ded5bb578404fc6c268e8f5ad1046e3a8b60512f40ece24250a82de636d2cfaf63ac4d3490012b82dc46
key 66 order lo_hi shift 0 rev False pack be mar

In [57]:
import os
for fn in ['bitplane_4_6_full.bin', 'bitplane_4_6.bin', 'bitplane_4_1.bin', 'bitplane_4_1_full.bin']:
    if os.path.exists(fn):
        print(fn, os.path.getsize(fn))
    else:
        print(fn, 'missing')


bitplane_4_6_full.bin 11617
bitplane_4_6.bin 8192
bitplane_4_1.bin 8192
bitplane_4_1_full.bin 11617


In [58]:
# Exhaustive search for nested headers in 6-bit decoded candidates with optional single-byte XOR post-transform
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
keys = [64, 66, 68, 69, 70, 71, 82, 86, 87, 117]

mapping_funcs = {
    'hi_lo': lambda b: ((b >> 4) & 0x7) * 8 + (b & 0x7),
    'lo_hi': lambda b: (b & 0x7) * 8 + ((b >> 4) & 0x7),
}
rev6 = {i: int('{:06b}'.format(i)[::-1], 2) for i in range(64)}
pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}
markers = {
    'gzip': b'\x1f\x8b\x08',
    'zip1': b'PK\x03\x04',
    'zip2': b'PK\x05\x06',
    'zip3': b'PK\x07\x08',
    '7z': b'7z\xbc\xaf\x27\x1c',
    'ctf=': b'ctf=',
    'flag{': b'flag{',
}

found = []
for key in keys:
    x = bytes(b ^ key for b in data)
    for order_name, sym_func in mapping_funcs.items():
        symbols = [sym_func(b) for b in x]
        if max(symbols) >= 64 or min(symbols) < 0:
            continue
        for shift in range(4):
            for rev in [False, True]:
                syms = symbols[shift:]
                if rev:
                    syms = [rev6[s] for s in syms]
                for pack_name, pack_func in pack_orders.items():
                    out = bytearray()
                    for i in range(0, len(syms) - (len(syms) % 4), 4):
                        v = pack_func(syms[i:i+4])
                        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
                    if not out:
                        continue
                    for xor_key in range(256):
                        xo = bytes(b ^ xor_key for b in out)
                        for name, patt in markers.items():
                            idx = xo.find(patt)
                            if idx != -1:
                                found.append((key, order_name, shift, rev, pack_name, xor_key, name, idx))
                                if len(found) >= 50:
                                    break
                        if len(found) >= 50:
                            break
                    if len(found) >= 50:
                        break
                if len(found) >= 50:
                    break
            if len(found) >= 50:
                break
        if len(found) >= 50:
            break
    if len(found) >= 50:
        break

print('found', len(found), 'matches')
for item in found[:50]:
    print(item)


found 34 matches
(64, 'hi_lo', 1, True, 'le', 30, 'gzip', 4372)
(64, 'lo_hi', 2, True, 'be', 99, 'gzip', 1891)
(64, 'lo_hi', 3, True, 'be', 97, 'gzip', 142)
(66, 'hi_lo', 0, False, 'le', 182, 'gzip', 6795)
(66, 'hi_lo', 3, False, 'le', 4, 'gzip', 4563)
(66, 'lo_hi', 1, False, 'be', 29, 'gzip', 6426)
(66, 'lo_hi', 3, True, 'be', 129, 'gzip', 6169)
(68, 'hi_lo', 0, True, 'be', 155, 'gzip', 1972)
(68, 'hi_lo', 1, False, 'be', 99, 'gzip', 6600)
(68, 'hi_lo', 1, False, 'le', 46, 'gzip', 6549)
(68, 'hi_lo', 2, False, 'le', 172, 'gzip', 5976)
(68, 'lo_hi', 3, False, 'be', 134, 'gzip', 474)
(69, 'hi_lo', 3, True, 'be', 228, 'gzip', 8089)
(69, 'lo_hi', 1, False, 'le', 81, 'gzip', 5264)
(69, 'lo_hi', 3, False, 'le', 164, 'gzip', 6402)
(70, 'hi_lo', 0, False, 'be', 198, 'gzip', 1483)
(70, 'hi_lo', 1, True, 'le', 33, 'gzip', 6138)
(71, 'hi_lo', 0, False, 'be', 38, 'gzip', 5014)
(71, 'hi_lo', 2, False, 'le', 47, 'gzip', 7840)
(71, 'lo_hi', 1, False, 'be', 150, 'gzip', 5801)
(82, 'hi_lo', 2, True, '

In [60]:
import gzip

fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
keys = [64, 66, 68, 69, 70, 71, 82, 86, 87, 117]

mapping_funcs = {
    'hi_lo': lambda b: ((b >> 4) & 0x7) * 8 + (b & 0x7),
    'lo_hi': lambda b: (b & 0x7) * 8 + ((b >> 4) & 0x7),
}
rev6 = {i: int('{:06b}'.format(i)[::-1], 2) for i in range(64)}
pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}

matches = [
    (64, 'hi_lo', 1, True, 'le', 30, 4372),
    (64, 'lo_hi', 2, True, 'be', 99, 1891),
    (64, 'lo_hi', 3, True, 'be', 97, 142),
    (66, 'hi_lo', 0, False, 'le', 182, 6795),
    (66, 'hi_lo', 3, False, 'le', 4, 4563),
    (66, 'lo_hi', 1, False, 'be', 29, 6426),
    (66, 'lo_hi', 3, True, 'be', 129, 6169),
    (68, 'hi_lo', 0, True, 'be', 155, 1972),
    (68, 'hi_lo', 1, False, 'be', 99, 6600),
    (68, 'hi_lo', 1, False, 'le', 46, 6549),
    (68, 'hi_lo', 2, False, 'le', 172, 5976),
    (68, 'lo_hi', 3, False, 'be', 134, 474),
    (69, 'hi_lo', 3, True, 'be', 228, 8089),
    (69, 'lo_hi', 1, False, 'le', 81, 5264),
    (69, 'lo_hi', 3, False, 'le', 164, 6402),
    (70, 'hi_lo', 0, False, 'be', 198, 1483),
    (70, 'hi_lo', 1, True, 'le', 33, 6138),
    (71, 'hi_lo', 0, False, 'be', 38, 5014),
    (71, 'hi_lo', 2, False, 'le', 47, 7840),
    (71, 'lo_hi', 1, False, 'be', 150, 5801),
    (82, 'hi_lo', 2, True, 'be', 234, 7054),
    (82, 'hi_lo', 3, False, 'be', 128, 4241),
    (82, 'lo_hi', 1, False, 'le', 70, 456),
    (86, 'hi_lo', 1, True, 'le', 70, 2330),
    (86, 'lo_hi', 1, True, 'be', 220, 6058),
    (86, 'lo_hi', 3, False, 'be', 131, 3980),
    (87, 'hi_lo', 1, False, 'be', 229, 5965),
    (87, 'hi_lo', 2, False, 'le', 194, 5457),
    (87, 'hi_lo', 3, True, 'be', 244, 265),
    (87, 'hi_lo', 3, True, 'le', 49, 3709),
    (117, 'hi_lo', 0, False, 'be', 177, 6499),
    (117, 'hi_lo', 1, True, 'le', 180, 2299),
    (117, 'hi_lo', 2, False, 'le', 3, 4380),
    (117, 'lo_hi', 3, True, 'le', 192, 5530),
]

for key, order_name, shift, rev, pack_name, xor_key, offset in matches:
    x = bytes(b ^ key for b in data)
    symbols = [mapping_funcs[order_name](b) for b in x]
    if rev:
        symbols = [rev6[s] for s in symbols[shift:]]
    else:
        symbols = symbols[shift:]
    out = bytearray()
    for i in range(0, len(symbols) - (len(symbols) % 4), 4):
        v = pack_orders[pack_name](symbols[i:i+4])
        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
    xo = bytes(b ^ xor_key for b in out)
    print('candidate', key, order_name, shift, rev, pack_name, xor_key, 'offset', offset, 'header', xo[offset:offset+16].hex())
    try:
        decompressed = gzip.decompress(xo[offset:])
        print('  decompressed len', len(decompressed))
        print('  decompressed prefix', decompressed[:128])
    except Exception as e:
        print('  decompress failed:', type(e).__name__, str(e))
    print('-'*80)


candidate 64 hi_lo 1 True le 30 offset 4372 header 1f8b083d3385d594b4e8b8b76d4813ee
  decompress failed: EOFError Compressed file ended before the end-of-stream marker was reached
--------------------------------------------------------------------------------
candidate 64 lo_hi 2 True be 99 offset 1891 header 1f8b08a2ef59c7611669cf5b897ad11b
  decompress failed: error Error -3 while decompressing data: invalid stored block lengths
--------------------------------------------------------------------------------
candidate 64 lo_hi 3 True be 97 offset 142 header 1f8b0888ce5a5261430bccc2e48db36e
  decompress failed: error Error -3 while decompressing data: invalid stored block lengths
--------------------------------------------------------------------------------
candidate 66 hi_lo 0 False le 182 offset 6795 header 1f8b08a0851ccc6bbc8b3c7c97fc59d1
  decompress failed: error Error -3 while decompressing data: invalid code lengths set
-------------------------------------------------------

In [61]:
# Test a single promising candidate to confirm gzip decompression behavior
import gzip
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()

mapping_funcs = {
    'hi_lo': lambda b: ((b >> 4) & 0x7) * 8 + (b & 0x7),
    'lo_hi': lambda b: (b & 0x7) * 8 + ((b >> 4) & 0x7),
}
rev6 = {i: int('{:06b}'.format(i)[::-1], 2) for i in range(64)}
pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}

candidate = (64, 'lo_hi', 3, True, 'be', 97, 142)
key, order_name, shift, rev, pack_name, xor_key, offset = candidate
x = bytes(b ^ key for b in data)
symbols = [mapping_funcs[order_name](b) for b in x]
if rev:
    symbols = [rev6[s] for s in symbols[shift:]]
else:
    symbols = symbols[shift:]
out = bytearray()
for i in range(0, len(symbols) - (len(symbols) % 4), 4):
    v = pack_orders[pack_name](symbols[i:i+4])
    out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
xo = bytes(b ^ xor_key for b in out)
print('len', len(xo))
print('header', xo[offset:offset+16].hex())
try:
    decompressed = gzip.decompress(xo[offset:])
    print('decompressed len', len(decompressed))
    print('prefix', decompressed[:64])
except Exception as e:
    print('decompress failed:', type(e).__name__, e)


len 8709
header 1f8b0888ce5a5261430bccc2e48db36e
decompress failed: error Error -3 while decompressing data: invalid stored block lengths


In [64]:
# Inspect the 6-bit symbol stream and ordered alphabet for the strongest XOR candidates
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
for key in [69, 70]:
    x = bytes(b ^ key for b in data)
    symbols = sorted(set(x))
    print('key', key, 'unique symbols', len(symbols))
    print(symbols)
    print('first 64 bytes:', x[:64].hex())
    print('freq sample:', sorted(((c, x.count(c)) for c in set(x)), key=lambda t: -t[1])[:16])
    print()

key 69 unique symbols 64
[0, 1, 2, 3, 4, 5, 6, 7, 16, 17, 18, 19, 20, 21, 22, 23, 32, 33, 34, 35, 36, 37, 38, 39, 48, 49, 50, 51, 52, 53, 54, 55, 64, 65, 66, 67, 68, 69, 70, 71, 80, 81, 82, 83, 84, 85, 86, 87, 96, 97, 98, 99, 100, 101, 102, 103, 112, 113, 114, 115, 116, 117, 118, 119]
first 64 bytes: 72166675167734747617733314733607214224754743001735175366643550434047651437432754422051724460175004646522461215765664677404257747
freq sample: [(101, 280), (85, 274), (100, 272), (84, 265), (68, 263), (69, 261), (86, 254), (117, 254), (116, 253), (87, 247), (102, 243), (71, 239), (70, 234), (119, 230), (103, 229), (118, 226)]

key 70 unique symbols 64
[0, 1, 2, 3, 4, 5, 6, 7, 16, 17, 18, 19, 20, 21, 22, 23, 32, 33, 34, 35, 36, 37, 38, 39, 48, 49, 50, 51, 52, 53, 54, 55, 64, 65, 66, 67, 68, 69, 70, 71, 80, 81, 82, 83, 84, 85, 86, 87, 96, 97, 98, 99, 100, 101, 102, 103, 112, 113, 114, 115, 116, 117, 118, 119]
first 64 bytes: 71156576157437777514703017703504224127764440031436145065673653404344

In [62]:
# Evaluate decompression for all found gzip candidates and report only successes or failure types
import gzip
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()

mapping_funcs = {
    'hi_lo': lambda b: ((b >> 4) & 0x7) * 8 + (b & 0x7),
    'lo_hi': lambda b: (b & 0x7) * 8 + ((b >> 4) & 0x7),
}
rev6 = {i: int('{:06b}'.format(i)[::-1], 2) for i in range(64)}
pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}

candidates = [
    (64, 'hi_lo', 1, True, 'le', 30, 4372),
    (64, 'lo_hi', 2, True, 'be', 99, 1891),
    (64, 'lo_hi', 3, True, 'be', 97, 142),
    (66, 'hi_lo', 0, False, 'le', 182, 6795),
    (66, 'hi_lo', 3, False, 'le', 4, 4563),
    (66, 'lo_hi', 1, False, 'be', 29, 6426),
    (66, 'lo_hi', 3, True, 'be', 129, 6169),
    (68, 'hi_lo', 0, True, 'be', 155, 1972),
    (68, 'hi_lo', 1, False, 'be', 99, 6600),
    (68, 'hi_lo', 1, False, 'le', 46, 6549),
    (68, 'hi_lo', 2, False, 'le', 172, 5976),
    (68, 'lo_hi', 3, False, 'be', 134, 474),
    (69, 'hi_lo', 3, True, 'be', 228, 8089),
    (69, 'lo_hi', 1, False, 'le', 81, 5264),
    (69, 'lo_hi', 3, False, 'le', 164, 6402),
    (70, 'hi_lo', 0, False, 'be', 198, 1483),
    (70, 'hi_lo', 1, True, 'le', 33, 6138),
    (71, 'hi_lo', 0, False, 'be', 38, 5014),
    (71, 'hi_lo', 2, False, 'le', 47, 7840),
    (71, 'lo_hi', 1, False, 'be', 150, 5801),
    (82, 'hi_lo', 2, True, 'be', 234, 7054),
    (82, 'hi_lo', 3, False, 'be', 128, 4241),
    (82, 'lo_hi', 1, False, 'le', 70, 456),
    (86, 'hi_lo', 1, True, 'le', 70, 2330),
    (86, 'lo_hi', 1, True, 'be', 220, 6058),
    (86, 'lo_hi', 3, False, 'be', 131, 3980),
    (87, 'hi_lo', 1, False, 'be', 229, 5965),
    (87, 'hi_lo', 2, False, 'le', 194, 5457),
    (87, 'hi_lo', 3, True, 'be', 244, 265),
    (87, 'hi_lo', 3, True, 'le', 49, 3709),
    (117, 'hi_lo', 0, False, 'be', 177, 6499),
    (117, 'hi_lo', 1, True, 'le', 180, 2299),
    (117, 'hi_lo', 2, False, 'le', 3, 4380),
    (117, 'lo_hi', 3, True, 'le', 192, 5530),
]

for key, order_name, shift, rev, pack_name, xor_key, offset in candidates:
    x = bytes(b ^ key for b in data)
    symbols = [mapping_funcs[order_name](b) for b in x]
    if rev:
        symbols = [rev6[s] for s in symbols[shift:]]
    else:
        symbols = symbols[shift:]
    out = bytearray()
    for i in range(0, len(symbols) - (len(symbols) % 4), 4):
        v = pack_orders[pack_name](symbols[i:i+4])
        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
    xo = bytes(b ^ xor_key for b in out)
    try:
        decompressed = gzip.decompress(xo[offset:])
        print('SUCCESS', key, order_name, shift, rev, pack_name, xor_key, offset, 'len', len(decompressed))
        print(decompressed[:128])
        break
    except Exception as e:
        print('FAIL', key, order_name, shift, rev, pack_name, xor_key, offset, type(e).__name__, str(e))


FAIL 64 hi_lo 1 True le 30 4372 EOFError Compressed file ended before the end-of-stream marker was reached
FAIL 64 lo_hi 2 True be 99 1891 error Error -3 while decompressing data: invalid stored block lengths
FAIL 64 lo_hi 3 True be 97 142 error Error -3 while decompressing data: invalid stored block lengths
FAIL 66 hi_lo 0 False le 182 6795 error Error -3 while decompressing data: invalid code lengths set
FAIL 66 hi_lo 3 False le 4 4563 error Error -3 while decompressing data: invalid distance too far back
FAIL 66 lo_hi 1 False be 29 6426 EOFError Compressed file ended before the end-of-stream marker was reached
FAIL 66 lo_hi 3 True be 129 6169 error Error -3 while decompressing data: invalid stored block lengths
FAIL 68 hi_lo 0 True be 155 1972 EOFError Compressed file ended before the end-of-stream marker was reached
FAIL 68 hi_lo 1 False be 99 6600 EOFError Compressed file ended before the end-of-stream marker was reached
FAIL 68 hi_lo 1 False le 46 6549 error Error -3 while decomp

In [63]:
# Search 6-bit decoded candidates + XOR for ZIP, MZ, 7z, ctf=, and flag{ markers
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
keys = [64, 66, 68, 69, 70, 71, 82, 86, 87, 117]

mapping_funcs = {
    'hi_lo': lambda b: ((b >> 4) & 0x7) * 8 + (b & 0x7),
    'lo_hi': lambda b: (b & 0x7) * 8 + ((b >> 4) & 0x7),
}
rev6 = {i: int('{:06b}'.format(i)[::-1], 2) for i in range(64)}
pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}
markers = {
    'PK03': b'PK\x03\x04',
    'PK05': b'PK\x05\x06',
    'PK07': b'PK\x07\x08',
    'MZ': b'MZ',
    '7z': b'7z\xbc\xaf\x27\x1c',
    'ctf=': b'ctf=',
    'flag{': b'flag{',
}

found = {name: [] for name in markers}
for key in keys:
    x = bytes(b ^ key for b in data)
    for order_name, sym_func in mapping_funcs.items():
        symbols = [sym_func(b) for b in x]
        if max(symbols) >= 64 or min(symbols) < 0:
            continue
        for shift in range(4):
            for rev in [False, True]:
                syms = symbols[shift:]
                if rev:
                    syms = [rev6[s] for s in syms]
                for pack_name, pack_func in pack_orders.items():
                    out = bytearray()
                    for i in range(0, len(syms) - (len(syms) % 4), 4):
                        v = pack_func(syms[i:i+4])
                        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
                    for xor_key in range(256):
                        xo = bytes(b ^ xor_key for b in out)
                        for name, patt in markers.items():
                            if len(found[name]) >= 20:
                                continue
                            idx = xo.find(patt)
                            if idx != -1:
                                found[name].append((key, order_name, shift, rev, pack_name, xor_key, idx))
    print('finished key', key)

for name, entries in found.items():
    print('===', name, '===')
    for entry in entries:
        print(entry)
    print()


finished key 64
finished key 66
finished key 68
finished key 69
finished key 70
finished key 71
finished key 82
finished key 86
finished key 87
finished key 117
=== PK03 ===

=== PK05 ===

=== PK07 ===

=== MZ ===
(64, 'hi_lo', 0, False, 'be', 12, 1460)
(64, 'hi_lo', 0, False, 'be', 86, 6812)
(64, 'hi_lo', 0, False, 'be', 88, 4301)
(64, 'hi_lo', 0, False, 'be', 120, 1421)
(64, 'hi_lo', 0, False, 'be', 127, 7663)
(64, 'hi_lo', 0, False, 'be', 136, 2810)
(64, 'hi_lo', 0, False, 'be', 154, 2856)
(64, 'hi_lo', 0, False, 'be', 155, 282)
(64, 'hi_lo', 0, False, 'be', 156, 354)
(64, 'hi_lo', 0, False, 'be', 161, 2705)
(64, 'hi_lo', 0, False, 'be', 177, 6777)
(64, 'hi_lo', 0, False, 'be', 181, 3407)
(64, 'hi_lo', 0, False, 'be', 183, 3940)
(64, 'hi_lo', 0, False, 'be', 185, 3250)
(64, 'hi_lo', 0, False, 'be', 194, 7839)
(64, 'hi_lo', 0, False, 'be', 197, 3260)
(64, 'hi_lo', 0, False, 'be', 200, 3343)
(64, 'hi_lo', 0, False, 'be', 205, 600)
(64, 'hi_lo', 0, False, 'be', 211, 5017)
(64, 'hi_lo',

In [65]:
# Search for ASCII markers directly in XORed bitplane_4_6_full bytes and in 6-bit decoded outputs
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
for key in [64, 66, 68, 69, 70, 71, 82, 86, 87, 117]:
    x = bytes(b ^ key for b in data)
    print('xor key', key, 'ctf=', x.find(b'ctf='), 'flag=', x.find(b'flag{'))

mapping_funcs = {
    'hi_lo': lambda b: ((b >> 4) & 0x7) * 8 + (b & 0x7),
    'lo_hi': lambda b: (b & 0x7) * 8 + ((b >> 4) & 0x7),
}
rev6 = {i: int('{:06b}'.format(i)[::-1], 2) for i in range(64)}
pack_orders = {
    'be': lambda s: ((s[0] << 18) | (s[1] << 12) | (s[2] << 6) | s[3]),
    'le': lambda s: ((s[3] << 18) | (s[2] << 12) | (s[1] << 6) | s[0]),
}

for key in [69,70]:
    x = bytes(b ^ key for b in data)
    for order_name, sym_func in mapping_funcs.items():
        symbols = [sym_func(b) for b in x]
        if max(symbols) >= 64 or min(symbols) < 0:
            continue
        for shift in range(4):
            for rev in [False, True]:
                syms = symbols[shift:]
                if rev:
                    syms = [rev6[s] for s in syms]
                for pack_name, pack_func in pack_orders.items():
                    out = bytearray()
                    for i in range(0, len(syms) - (len(syms) % 4), 4):
                        v = pack_func(syms[i:i+4])
                        out.extend([(v >> 16) & 0xFF, (v >> 8) & 0xFF, v & 0xFF])
                    print('key', key, order_name, 'shift', shift, 'rev', rev, 'pack', pack_name,
                          'ctf=', out.find(b'ctf='), 'flag=', out.find(b'flag{'))


xor key 64 ctf= -1 flag= -1
xor key 66 ctf= -1 flag= -1
xor key 68 ctf= -1 flag= -1
xor key 69 ctf= -1 flag= -1
xor key 70 ctf= -1 flag= -1
xor key 71 ctf= -1 flag= -1
xor key 82 ctf= -1 flag= -1
xor key 86 ctf= -1 flag= -1
xor key 87 ctf= -1 flag= -1
xor key 117 ctf= -1 flag= -1
key 69 hi_lo shift 0 rev False pack be ctf= -1 flag= -1
key 69 hi_lo shift 0 rev False pack le ctf= -1 flag= -1
key 69 hi_lo shift 0 rev True pack be ctf= -1 flag= -1
key 69 hi_lo shift 0 rev True pack le ctf= -1 flag= -1
key 69 hi_lo shift 1 rev False pack be ctf= -1 flag= -1
key 69 hi_lo shift 1 rev False pack le ctf= -1 flag= -1
key 69 hi_lo shift 1 rev True pack be ctf= -1 flag= -1
key 69 hi_lo shift 1 rev True pack le ctf= -1 flag= -1
key 69 hi_lo shift 2 rev False pack be ctf= -1 flag= -1
key 69 hi_lo shift 2 rev False pack le ctf= -1 flag= -1
key 69 hi_lo shift 2 rev True pack be ctf= -1 flag= -1
key 69 hi_lo shift 2 rev True pack le ctf= -1 flag= -1
key 69 hi_lo shift 3 rev False pack be ctf= -1 flag= 

In [66]:
# Search all single-byte XOR keys for the 64-symbol pattern in bitplane_4_6_full.bin
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
keys_64 = []
for key in range(256):
    x = bytes(b ^ key for b in data)
    if len(set(x)) == 64:
        keys_64.append(key)
print('keys with 64 symbols:', keys_64)
print('count', len(keys_64))
# Show the sorted unique values for the first few keys.
for key in keys_64[:10]:
    x = bytes(b ^ key for b in data)
    print('key', key, 'uniq', sorted(set(x)))


keys with 64 symbols: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 21

In [69]:
# Check whether bitplane_4_6_full.bin raw bytes already use the 64-symbol alphabet
fn = 'bitplane_4_6_full.bin'
data = open(fn, 'rb').read()
uniq = sorted(set(data))
print('raw unique symbol count', len(uniq))
print(uniq)
print('group counts:', {g: sum(1 for b in uniq if b >> 4 == g) for g in range(16)})


raw unique symbol count 64
[0, 1, 2, 3, 4, 5, 6, 7, 16, 17, 18, 19, 20, 21, 22, 23, 32, 33, 34, 35, 36, 37, 38, 39, 48, 49, 50, 51, 52, 53, 54, 55, 64, 65, 66, 67, 68, 69, 70, 71, 80, 81, 82, 83, 84, 85, 86, 87, 96, 97, 98, 99, 100, 101, 102, 103, 112, 113, 114, 115, 116, 117, 118, 119]
group counts: {0: 8, 1: 8, 2: 8, 3: 8, 4: 8, 5: 8, 6: 8, 7: 8, 8: 0, 9: 0, 10: 0, 11: 0, 12: 0, 13: 0, 14: 0, 15: 0}


In [98]:
# Run the reconstruction scoring script and print results
import subprocess
import sys
from pathlib import Path

script = Path('reconstruct.py')
print('script exists:', script.exists())
if script.exists():
    result = subprocess.run([sys.executable, str(script)], capture_output=True, text=True)
    print('returncode', result.returncode)
    print(result.stdout)
    print(result.stderr)
else:
    print('reconstruct.py missing')

script exists: True


KeyboardInterrupt: 

In [99]:
# Targeted byte-alignment candidate search with scoring around ctf=
import math
from collections import Counter
import torch

PATTERNS = {
    'ctf=': b'ctf=',
    'flag{': b'flag{',
    'MZ': b'MZ',
    'gzip': b'\x1f\x8b',
    'PK': b'PK',
}

BLOCKS = [1, 2, 4, 8, 16]
MAX_SHIFT = 32


def load_raw():
    obj = torch.load('ode.pt', map_location='cpu', weights_only=False)
    raw6 = obj['model']['transformer.h.6.mlp.c_proj.weight'].numpy().tobytes()
    raw7 = obj['model']['transformer.h.7.mlp.c_proj.weight'].numpy().tobytes()
    return raw6, raw7


def deinterleave(a: bytes, b: bytes, block_size: int, shift: int, reverse: bool) -> bytes:
    if reverse:
        a, b = b, a
    if shift:
        a = a[shift:]
        b = b[:len(a)]
    length = min(len(a), len(b))
    out = bytearray()
    i = 0
    while i < length:
        out.extend(a[i : i + block_size])
        out.extend(b[i : i + block_size])
        i += block_size
    return bytes(out)


def longest_ascii_run(data: bytes, min_len: int = 4) -> int:
    best = 0
    curr = 0
    for c in data:
        if 32 <= c < 127:
            curr += 1
            if curr > best:
                best = curr
        else:
            curr = 0
    return best if best >= min_len else 0


def printable_ratio(data: bytes) -> float:
    return sum(32 <= c < 127 for c in data) / len(data) if data else 0.0


def entropy(data: bytes) -> float:
    if not data:
        return 0.0
    counts = Counter(data)
    n = len(data)
    return -sum((cnt / n) * math.log2(cnt / n) for cnt in counts.values())


def markers(blob: bytes) -> dict[str, int]:
    return {name: blob.find(pattern) for name, pattern in PATTERNS.items()}


raw6, raw7 = load_raw()
print('loaded raw6', len(raw6), 'raw7', len(raw7))

candidates = []
for block in BLOCKS:
    for shift in range(MAX_SHIFT + 1):
        for reverse in (False, True):
            blob = deinterleave(raw6, raw7, block_size=block, shift=shift, reverse=reverse)
            m = markers(blob)
            if m['ctf='] == -1:
                continue
            ctf = m['ctf=']
            region = blob[max(0, ctf - 64) : ctf + 256]
            candidates.append({
                'block': block,
                'shift': shift,
                'reverse': reverse,
                'ctf': ctf,
                'marker_count': sum(1 for v in m.values() if v != -1),
                'print_ratio': printable_ratio(region),
                'ascii_run': longest_ascii_run(region),
                'entropy': entropy(region),
                'markers': m,
                'window': region[:128],
            })

if not candidates:
    print('no candidates with ctf= found')
else:
    candidates.sort(key=lambda c: (-c['marker_count'], -c['print_ratio'], -c['ascii_run'], c['entropy'], c['ctf']))
    print('candidates:', len(candidates))
    for idx, cand in enumerate(candidates[:20], 1):
        print(
            f"{idx:02d}. block={cand['block']} shift={cand['shift']} reverse={cand['reverse']} "
            f"ctf={cand['ctf']} markers={cand['marker_count']} print={cand['print_ratio']:.4f} "
            f"ascii={cand['ascii_run']} ent={cand['entropy']:.4f} "
            f"MZ={cand['markers']['MZ']} gzip={cand['markers']['gzip']} PK={cand['markers']['PK']} flag={cand['markers']['flag{']}"
        )
        print(cand['window'].hex())
        print()

loaded raw6 6553600 raw7 6553600
candidates: 156
01. block=16 shift=6 reverse=True ctf=13014242 markers=4 print=0.4031 ascii=9 ent=6.7529 MZ=38292 gzip=51964 PK=128067 flag=-1
823ee63cf5fbd73c9a04273d36311845013c332e61bd3061163c93a29ebd5ebcedcfc43d8332173d0c6a2b3d675665332e3df57d553d42e1903bd4699cbc153e6374663dc5ca14be6fcc853caf3fd83e9bbcb27f9a3dd3ecd3bacfe415bebebd6cec093e8ecc01bb7522a9bd6994e39ddabc9862853cd4daabbdc3ac6c3a3f3e

02. block=16 shift=5 reverse=True ctf=13014243 markers=4 print=0.4031 ascii=8 ent=6.7529 MZ=38292 gzip=51964 PK=128068 flag=-1
823ee63cf5fbd73c9a04273d361845013c332e61bd3061163c93a29ebd315ebcedcfc43d8332173d0c6a2b3d6765332e3df57d553d42e1903bd4699cbc56153e6374663dc5ca14be6fcc853cafd83e9bbcb27f9a3dd3ecd3bacfe415be3fbebd6cec093e8ecc01bb7522a9bd69e39ddabc9862853cd4daabbdc3ac6c3a943f3e

03. block=8 shift=7 reverse=True ctf=13014241 markers=4 print=0.4031 ascii=7 ent=6.7529 MZ=38284 gzip=51964 PK=128066 flag=-1
823ee63cf5fbd71845013c332e61bd3c9a04273d36315e3061163c9